# Soft Sensing of Intracellular States for CHO Cell Bioprocessing with Ensemble Kalman Filters

This repository accompanies the manuscript  
**“Soft Sensing of Intracellular States for CHO Cell Bioprocessing with Ensemble Kalman Filters”**,  
which investigates the inference of unmeasured intracellular states in CHO cell cultures using a Bayesian state estimation framework.

Direct measurement of intracellular nucleotide sugar donors (NSDs) is analytically challenging and incompatible with routine process monitoring. In this work, an Ensemble Kalman Filter (EnKF) is employed to dynamically infer NSD concentrations from standard extracellular measurements, enabling earlier and quality-relevant insight into glycosylation-critical intracellular dynamics.

The repository provides:
- A fully reproducible Jupyter notebook implementing the EnKF framework  
- Supporting utilities for simulation, data handling, and result storage  
- Scripts to reproduce all figures and analyses reported in the manuscript  

The code is organised to ensure portability and reproducibility. All file paths are defined relative to the repository root, and no absolute or machine-specific paths are used.



## 1. Package imports and result handling utilities

This section introduces the Python packages used throughout the study and defines utility functions for handling file paths and storing intermediate results. The environment relies on standard scientific computing libraries commonly used in bioprocess modelling, numerical simulation, and data visualisation.

In [2]:
# Core scientific computing
import numpy as np
import pandas as pd

# Numerical methods
from scipy.integrate import solve_ivp
import scipy.integrate as scp
from scipy.linalg import inv
from sklearn.metrics import mean_squared_error


# Plotting
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns

# Utilities
import os
import sys
import copy
import csv
import pickle
from pathlib import Path

# Progress bar (useful for ensemble loops)
from tqdm import tqdm

# Notebook display helpers (optional)
from IPython.display import display, HTML


In [ ]:
"""
Utility functions for saving and loading intermediate results.

All paths are defined relative to the repository root to ensure
portability and reproducibility when the code is run on different
machines or operating systems.
"""

from pathlib import Path
import pickle


def save_pkl(item, fname, base_dir):
    """
    Save a Python object to a pickle file in the given directory.
    """
    file_path = Path(base_dir) / fname
    file_path.parent.mkdir(parents=True, exist_ok=True)
    with open(file_path, "wb") as f:
        pickle.dump(item, f)

    print("✅ Saved to:", file_path)
    return file_path


def load_pkl(fname, base_dir):
    """
    Load a Python object from a pickle file in the given directory.
    """
    file_path = Path(base_dir) / fname
    print("🔎 Attempting to load:", file_path)

    if not file_path.exists():
        raise FileNotFoundError(
            f"File not found: {file_path}\n"
            "Ensure the results have been generated before loading."
        )

    with open(file_path, "rb") as f:
        obj = pickle.load(f)

    print("✅ Loaded from:", file_path)
    return obj

## 2. Experimental datasets and data loading

This section defines the experimental datasets used in this study and provides utility functions for loading the corresponding measurement data. Four independent fed-batch CHO cell culture experiments (P1–P4), each employing distinct feeding strategies, are considered.

Each dataset is stored as a single Excel file containing:
- Extracellular metabolite measurements (used for EnKF assimilation)
- Intracellular nucleotide sugar donor (NSD) measurements (used for validation only)


In [ ]:
# ---------------------------------------------------------------------
# Repository root (auto-detected)
# ---------------------------------------------------------------------
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "data").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

# ---------------------------------------------------------------------
# Data directory (relative to repository root)
# ---------------------------------------------------------------------
# Expected structure:
# data/
# └── raw/
#     ├── P1.xlsx
#     ├── P2.xlsx
#     ├── P3.xlsx
#     └── P4.xlsx
DATA_DIR = REPO_ROOT / "data" / "raw"

# Results directory
BASE_RESULTS_DIR = REPO_ROOT / "results"
BASE_RESULTS_DIR.mkdir(exist_ok=True)


# ---------------------------------------------------------------------
# Dataset configuration
# ---------------------------------------------------------------------
# Each dataset is defined by:
# - Excel filename
# - Sheet names for extracellular metabolites and NSDs
#
# This configuration layer avoids hard-coding filenames and
# allows flexible selection of subsets of experiments.
DATASETS_ALL = {
    f"P{i}": {
        "file": DATA_DIR / f"P{i}.xlsx",
        "sheets": {
            "met": "Metabolites",  # extracellular measurements
            "nsd": "NSD"           # intracellular NSD measurements
        }
    }
    for i in range(1, 5)
}


def select_datasets(*names, datasets_all=DATASETS_ALL):
    """
    Select a subset of datasets by name.

    Parameters
    ----------
    *names : str
        Dataset identifiers (e.g. 'P1', 'P3').
        If no names are provided, all available datasets are returned.
    datasets_all : dict
        Dictionary containing all dataset configurations.

    Returns
    -------
    dict
        Filtered dataset configuration dictionary.

    Examples
    --------
    DATASETS = select_datasets("P1")
    DATASETS = select_datasets("P2", "P4")
    DATASETS = select_datasets()  # returns all datasets (P1–P4)
    """
    if len(names) == 0:
        return dict(datasets_all)

    names = [str(n).upper() for n in names]

    missing = [n for n in names if n not in datasets_all]
    if missing:
        raise KeyError(
            f"Unknown dataset(s): {missing}. "
            f"Available datasets: {list(datasets_all.keys())}"
        )

    return {n: datasets_all[n] for n in names}


def load_dataset(name, datasets_cfg=DATASETS_ALL):
    """
    Load measurement data for a single dataset.

    Parameters
    ----------
    name : str
        Dataset identifier (e.g. 'P1').
    datasets_cfg : dict
        Dataset configuration dictionary.

    Returns
    -------
    dict
        Dictionary containing:
        - set_meas : extracellular measurements (numpy array)
        - NSD_meas : intracellular NSD measurements (numpy array)
        - set_meas_errorbar : extracellular measurement uncertainties
        - NSD_meas_errorbar : NSD measurement uncertainties
        - met_df : raw metabolite DataFrame
        - nsd_df : raw NSD DataFrame
    """
    name = str(name).upper()
    cfg = datasets_cfg[name]
    file_path = cfg["file"]

    if not file_path.exists():
        raise FileNotFoundError(
            f"Data file not found: {file_path}\n"
            "Ensure the Excel files are placed in data/raw/."
        )

    # -----------------------------------------------------------------
    # Load Excel sheets
    # -----------------------------------------------------------------
    met_df = pd.read_excel(file_path, sheet_name=cfg["sheets"]["met"])
    nsd_df = pd.read_excel(file_path, sheet_name=cfg["sheets"]["nsd"])

    # -----------------------------------------------------------------
    # Extract numerical measurement arrays
    # -----------------------------------------------------------------
    # Assumed data layout:
    # - First two rows contain headers / metadata
    # - Measurements and error bars alternate by column
    #
    # Extracellular metabolites:
    #   columns: [value, error, value, error, ...]
    set_meas = met_df.iloc[2:, 3::2].to_numpy()
    set_meas_errorbar = met_df.iloc[2:, 4::2].to_numpy()

    # Intracellular NSDs:
    NSD_meas = nsd_df.iloc[2:, 1::2].to_numpy()
    NSD_meas_errorbar = nsd_df.iloc[2:, 2::2].to_numpy()

    return {
        "set_meas": set_meas,
        "NSD_meas": NSD_meas,
        "set_meas_errorbar": set_meas_errorbar,
        "NSD_meas_errorbar": NSD_meas_errorbar,
        "met_df": met_df,
        "nsd_df": nsd_df,
    }


# ---------------------------------------------------------------------
# Dataset selection for the current analysis
# ---------------------------------------------------------------------
# Here, all four experimental conditions are included.
DATASETS = select_datasets("P1", "P2", "P3", "P4")

## 3. State definitions and simulation time grid

This section defines the model state vector and associated labels used for visualisation and performance metrics. The state ordering is kept fixed throughout the notebook to ensure consistent indexing across model simulation, Ensemble Kalman Filter updates, and post-processing.

A uniform time grid is defined over the 12-day cultivation horizon using a 0.01 h resolution. 




In [55]:
# ---------------------------------------------------------------------
# State vector definition
# ---------------------------------------------------------------------
# The state ordering is fixed and used consistently across:
# - model simulation (ODE integration)
# - EnKF state propagation and measurement update
# - plotting and RMSE calculations
#
# The first states correspond to extracellular variables and product,
# followed by intracellular metabolites (including NSDs).
state_name = [
    "Xv", "mAb", "Gal", "Urd", "Glc", "Amm", "Gln", "Lac",
    "Asn", "Glu",
    "UDPGal", "UDPGalNAc", "UDPGlc", "UDPGlcNAc",
    "GDPMan", "GDPFuc", "CMPNeu5Ac"
]

# Total number of states in the model
state_num = len(state_name)


# ---------------------------------------------------------------------
# Labels for figures and summary metrics
# ---------------------------------------------------------------------
# These labels are aligned with `state_name` by index.
# They are used for axis titles and error metric reporting.
axis_name = [
    r"Viable Cell Density (cell L$^{-1}$)",
    r"mAb Titre (mg L$^{-1}$)",
    r"Galactose Concentration (mM)",
    r"Uridine Concentration (mM)",
    r"Glucose Concentration (mM)",
    r"Ammonia Concentration (mM)",
    r"Glutamine Concentration (mM)",
    r"Lactate Concentration (mM)",
    r"Asparagine Concentration (mM)",
    r"Glutamate Concentration (mM)",
    r"UDP-Gal Concentration (mM)",
    r"UDP-GalNAc Concentration (mM)",
    r"UDP-Glc Concentration (mM)",
    r"UDP-GlcNAc Concentration (mM)",
    r"GDP-Man Concentration (mM)",
    r"GDP-Fuc Concentration (mM)",
    r"CMP-Neu5Ac Concentration (mM)"
]

rmse_name = [
    r"RMSE-Viable Cell Density (cell L$^{-1}$)",
    r"RMSE-mAb (mg L$^{-1}$)",
    r"RMSE-Gal (mM)",
    r"RMSE-Urd (mM)",
    r"RMSE-Glucose (mM)",
    r"RMSE-Ammonia (mM)",
    r"RMSE-Gln (mM)",
    r"RMSE-Lac (mM)",
    r"RMSE-Asn (mM)",
    r"RMSE-Glu (mM)",
    r"RMSE-UDPGal (mM)",
    r"RMSE-UDPGalNAc (mM)",
    r"RMSE-UDPGlc (mM)",
    r"RMSE-UDPGlcNAc (mM)",
    r"RMSE-GDPMan (mM)",
    r"RMSE-GDPFuc (mM)",
    r"RMSE-CMPNeu5Ac (mM)"
]


# ---------------------------------------------------------------------
# Measurement vector size (variables assimilated by the EnKF)
# ---------------------------------------------------------------------
# In this study, the EnKF is updated using routinely measurable
# extracellular variables. Intracellular NSDs are not assimilated and
# are reserved for validation.
#
# Here, `meas_num` defines the number of variables included in the
# measurement update. The default choice includes:
#   Xv, mAb, Gal, Urd, Glc, Amm, Gln, Lac
#
# Asparagine (Asn) may be treated separately in later sections depending
# on the specific estimation scenario.
measured_states = ["Xv", "mAb", "Gal", "Urd", "Glc", "Amm", "Gln", "Lac"]
meas_num = len(measured_states)


# ---------------------------------------------------------------------
# Simulation time grid
# ---------------------------------------------------------------------
# The cultivation horizon is 12 days = 288 hours.
# A 0.01 h resolution (36 s) is used to:
# - represent bolus/sampling events as discrete time points
# - retain a fine grid for numerical integration and filtering
DT = 0.01
T_END = 288.0

# Time points used across the notebook (starts at 0.01 h to match original indexing)
time = np.arange(DT, T_END + DT, DT)

# Step size array used by some integration/update routines
step_len = np.full((len(time),), DT)
steps_n = len(step_len)

# Initial time point list (retained for compatibility with original solver logic)
t_l_0 = [0.0]


## 4. Feeding and sampling schedule construction

This section defines the discrete event schedule used to represent routine feeding, sampling, and bolus additions over the cultivation horizon. Inputs are implemented on the same uniform time grid defined previously, such that each event occurs at a specific time point and is represented as a single-step impulse.

A common baseline schedule is shared across all experiments for the volumetric feed and sampling profiles. Dataset-specific bolus additions of galactose and uridine are then applied at predefined time points to reflect the experimentally implemented feeding strategies.


In [56]:
# ---------------------------------------------------------------------
# Event timing on the existing time grid
# ---------------------------------------------------------------------
# This section assumes the following have already been defined:
# - DT : time step size (h)
# - time : global time vector (h), length = steps_n
# - steps_n : number of time steps
# - DATASETS : selected dataset configuration (e.g. P1–P4)

# Common feed pulses (Fin) applied across datasets.
# Each pulse is represented as a single time-step impulse on the DT grid.
FIN_PULSES = [48.02, 96.02, 144.02, 192.02, 240.02]  # hours

# Sampling / withdrawal pulses (Fout) applied across datasets.
# Keys are event times (h), values are withdrawal magnitudes (units consistent with your model).
FOUT_PULSES = {
    48.01: 0.63,
    72.01: 0.37,
    96.01: 0.30,
    96.03: 0.36,   # associated with gal/urd bolus handling in original workflow
    120.01: 0.24,
    144.01: 0.26,
    144.03: 0.10,  # gal/urd-related event
    168.01: 0.54,
    192.01: 0.23,
    192.03: 0.10,  # gal/urd-related event
    216.01: 0.53,
    240.01: 0.23,
    240.03: 0.10,  # gal/urd-related event
    264.00: 0.53
}

# Dataset-specific galactose and uridine bolus doses at selected feed times.
# Values are tuples (Gal_bolus, Urd_bolus). Units should be consistent with model input definitions.
BOLUS_DOSES = {
    "P1": {
        96.02:  (79.35, 15.87),
        144.02: (15.38, 3.08),
        192.02: (10.99, 2.20),
        240.02: (248.29, 49.66)
    },
    "P2": {
        96.02:  (4.27, 0.85),
        144.02: (168.34, 33.67),
        192.02: (37.72, 7.54),
        240.02: (11.35, 2.27)
    },
    "P3": {
        96.02:  (5.19, 1.04),
        144.02: (3.11, 0.62),
        192.02: (235.29, 47.06),
        240.02: (249.94, 49.99)
    },
    "P4": {
        96.02:  (21.91, 4.38),
        144.02: (6.41, 1.28),
        192.02: (233.46, 46.69),
        240.02: (3.97, 0.79)
    }
}

def hr_to_idx(t_hr: float, dt: float = DT) -> int:
    """Convert culture time in hours to array index on the 0.01 h grid."""
    return int(round(t_hr / dt))  # time starts at 0.01 h in your grid

def build_schedule(dataset_name: str, n_steps: int = steps_n, dt: float = DT):
    """
    Construct Fin, Fout, Gal_feed, Urd_feed arrays for a given dataset.
    All arrays have length n_steps and align with the global time vector.
    """
    Fin = np.zeros(n_steps)
    Fout = np.zeros(n_steps)
    Gal_feed = np.zeros(n_steps)
    Urd_feed = np.zeros(n_steps)

    # Fin pulses
    for t in FIN_PULSES:
        Fin[hr_to_idx(t, dt)] = 1.0

    # Fout pulses
    for t, val in FOUT_PULSES.items():
        Fout[hr_to_idx(t, dt)] = val

    # Gal and Urd boluses (dataset specific)
    doses = BOLUS_DOSES[dataset_name]
    for t, (gal_val, urd_val) in doses.items():
        idx = hr_to_idx(t, dt)
        Gal_feed[idx] = gal_val
        Urd_feed[idx] = urd_val

    return Fin, Fout, Gal_feed, Urd_feed


all_inputs = {}
for name in DATASETS.keys():  # P1–P4
    Fin, Fout, Gal_feed, Urd_feed = build_schedule(name)
    all_inputs[name] = {
        "Fin": Fin,
        "Fout": Fout,
        "Gal_feed": Gal_feed,
        "Urd_feed": Urd_feed
    }



## 5. Bioreactor volume trajectory

Culture volume evolves according to the net inlet and outlet flow rates:

$$
\frac{dV}{dt} = F_{in}(t) - F_{out}(t)
$$

where $F_{in}(t)$ and $F_{out}(t)$ are defined by the dataset-specific feeding and sampling schedule.

For each dataset (P1–P4), we:
1. Build its schedule to obtain $F_{in}(t)$ and $F_{out}(t)$.
2. Integrate the volume ODE over the full 288 h horizon using the same 0.01 h grid.
3. Store the resulting volume trajectory for simulation and plotting.

The numerical setup matches the original scripts to ensure identical outputs.


In [57]:
# ---------------------------------------------------------------------
# Volume integration (original implementation retained)
# ---------------------------------------------------------------------
# This section computes bioreactor volume trajectories for each dataset
# using the net flow balance:
#     dV/dt = Fin - Fout
#
# The implementation below is intentionally kept identical to the original
# scripts to ensure identical numerical behaviour and outputs.


def _to_scalar(x):
    """Safely convert 0-d or 1-element arrays to a Python float."""
    return np.asarray(x).reshape(-1)[0].item()

def volume_integration(init_volume, Fin, Fout, step_len):
    """
    Computes volume integration over time using ODE.

    Parameters:
        init_volume (float): Initial volume.
        Fin (array): Inlet flow rate (L/hr).
        Fout (array): Outlet flow rate (L/hr).
        step_len (array): Time step array (hours).

    Returns:
        np.array: Integrated volume over time with length len(Fin)+1.
    """
    # ODE right-hand side for the volume balance.
    # fin and fout are treated as constant over the current integration step.
    def volume_model(t, state, fin, fout):
        return np.array([fin - fout], dtype="float64")

    current_volume = _to_scalar(init_volume)
    volumes = [current_volume]

    # Stepwise integration over the discrete time grid.
    # Each iteration integrates from the current volume over a single dt step.
    for step_i in tqdm(range(len(Fin)), desc="Integrating Volume", leave=False):
        fin = _to_scalar(Fin[step_i])
        fout = _to_scalar(Fout[step_i])

        # LSODA is used for robust integration across stiff/non-stiff regimes.
        # nsteps is set high to prevent step limit issues.
        ode = scp.ode(volume_model).set_integrator("lsoda", nsteps=3000)
        ode.set_initial_value(current_volume, 0.0).set_f_params(fin, fout)

        new_volume = ode.integrate(ode.t + _to_scalar(step_len[step_i]))
        current_volume = _to_scalar(new_volume)  # update volume as scalar
        volumes.append(current_volume)

    return np.array(volumes)


# ---------------------------------------------------------------------
# Dataset-specific initial volumes
# ---------------------------------------------------------------------
# If all datasets share the same initial volume, this dict could be replaced
# by a single scalar; it is kept here for flexibility.
initial_volumes_P = {
    "P1": 0.1,
    "P2": 0.1,
    "P3": 0.1,
    "P4": 0.1
}

# ---------------------------------------------------------------------
# Compute volume trajectories for all selected datasets
# ---------------------------------------------------------------------
# Volume results are stored in a dictionary keyed by dataset name.
volume_results_P = {}


for name in DATASETS.keys():  # P1–P4
    Fin, Fout, Gal_feed, Urd_feed = build_schedule(name)

    init_volume = initial_volumes_P.get(name, 0.1)

    volume_results_P[name] = volume_integration(
        init_volume=init_volume,
        Fin=Fin,
        Fout=Fout,
        step_len=step_len
    )

    print(f"Completed volume integration for {name}")

print("All datasets processed")



Completed volume integration for P1


Completed volume integration for P2


Completed volume integration for P3


Completed volume integration for P4
All datasets processed


## 6. Process model parameters and rate definitions

This section specifies the fixed parameter values used in the mechanistic process model and defines the set of kinetic rates required for simulation and state estimation. Parameters include growth and death kinetics, extracellular metabolite yields, feed compositions, and the intracellular NSD pathway kinetics.


In [58]:
V_cell = 1.123e-12;   # [=] L/v.cell
MW = 165174;    # [=] g_prod/mol_prod
Acomp = 9.915651e-9;
    
SP = 71;
K_c = 2.572248E-6;
t_i = 176.0905;
t_d = 290.347;
#ITAE = 530-2.76709;

############################################## mu and Xv related ##############################################

mu_max = 0.065;
mu_d_max = 0.015;

Klysis = 0.5; 

Kglc = 14.0378;
Klac = 0.00001;
Kasn = 2.62371;
Kglu = 0.000001;
Kgln = 0.00000454277;

KIlac = 1000;
KIamm = 3.16935;
KIurd = 41.0875;
KIgal = 1000;

Kd_amm = 14.2830;
Kd_urd = 27.8752;

#Kd_gal =10e3;
#Kd_urd_gal =1E-6;
#Kd_asn = 7.8805E-02;

# regulator-constants      
Kgal = 18.2317;
Kurd = 7.00810;

m_lac = 1.87253e-10;#mmol/cell/h
m_glc = 3.43293e-11;
Kc_gal = 5.27033;

Lac_max_1 = 21.1983;
Lac_max_2 = 16;
factor = 0.347987069;
ng = 1

In [21]:
# Metabolism
# Concentration of each metabolite and amino acid in the feed (mM)
# NOT ESTIMATED

Lac_feed = 0.;

Glc_feed = 144.37;
Glu_feed = 12.19;
Asp_feed = 51.95;
Arg_feed = 9.16;
Asn_feed = 26.99;
Lys_feed = 16.64;
Pro_feed = 10.18;

# Phil feeds
Gln_feed = 0.;
Amm_feed = 0.06;

# yield of biomass on the metabolite (cell/mmol) 
Yx_glc = 1.0115e9;
Yx_gln = 4.64127e9;
Yx_glu = 1.45647e10;
Yx_lac = 5.45539e7;
Yx_amm = 2.36299e9;
Yx_gal = 1.38498e8;
Yx_urd = 1.61202e9;
Yx_asn = 7.6824e8;
Yx_asp = 3.59E9;
Yx_arg = 2.64E10;
Yx_lys = 1.75E10;
Yx_pro = 3.26e11;

# YmetA_metB Yield of the first metabolite (metA) from the second (metB) (mmol/mmol)
Ygln_glu = 0.;
Ygln_asn = 0.;
Ygln_amm = 0.104524;
Ylac_glc = 1.56;  # Yoon et. al 2003 ''Effect of low culture temperature on specific productivity,transcription level, and heterogeneity of erythropoietin in Chinese hamster ovary cells
Yasp_asn = 0.126;
Yarg_glu = 0.007;
Ylys_glu = 0.116;
Ypro_glu = 1.;
Yamm_urd = 2.;
Yasn_asp = 0.1;

######  MAB MODEL   #######################
YmAb_Xv = 4.12718e-10;
YmAb_mu = 3.38956e-9;

In [22]:
####################################### NSD #######################################

#gln intracellular
f2 = 0.0222435;

# maximum velocity (mmol·L-1·h-1) of the main NSD pathway 
Vmax1 = 0.921507;
Vmax2 = 0.0169968;
Vmax2b = 59.4891;              
Vmax3 = 0.0550887;
Vmax4 = 0.0265253;
Vmax5 = 0.0001;
Vmax6 = 5.1304;
Vmax7 = 4.59677;

# constants for each reaction (mM) of the main NSD pathway 
Kdf_E1_gln = 0.418760;
Kdf_E2b = 0.0248298;
Kdf_E4 = 2.31278;
Kdf_E5 = 0.0269656;
Kdf_E6 = 0.0163559;
Kdf_E7 = 0.994547;
Kdf_Glc_UDPGlc = 78.1241;
Kdf_Glc_GDPMan = 50.;

# inhibition constants for each reaction (mM) of the main NSD pathway 
KiE2A = 1.04504e-6;
KiE2B = 92.1059;
KiE2C = 0.0132697;
KiE2D = 2.66102e-6;
KiE5 = 1000.;
KiE6A = 1.10182e-7;
KiE6B = 4.57309;
KiE6C = 4.83505e-6;
Ki_E7 = 0.0164192;

# maximum velocity of the reactions (mmol·L-1·h-1) of the rates due to gal/urd feeding
Vmax1u = 0.147995;
Vmax2u = 0.0451806;
Vmax4u = 0.0127551;
Vmax6u = 5.34270;
Vmax6g = 40.8965;

# constants for each reaction (mM) of the rates due to gal/urd feeding
K1u = 6.08196;
K2u = 13.6332;
K4u = 6.24826;
K6u = 0.438499;
K6g = 0.600019;

# reaction inhibition constants (mM) of the rates due to gal/urd feeding
Ki6_urd = 0.000911002;
Ki6_glc = 0.292793;
Ki6_gal = 99.6298;
Ki6_ugal = 0.01;#(not bound)

# maximum velocity of the reactions (mmol·L-1·h-1) of the sink rates
Vmax6_sink = 7.30429;
Vmax7_sink = 10.9370;
Vmax1_sink = 25.4859;

# constants for each reaction (mM) of the sink rates
K6_sink = 0.128756;
K7_sink = 8.87794;
K1_sink = 0.0406881;

# reaction inhibition constants (mM) of the sink rates
Ki1_sink = 0.000120640;

# Fout Monod type constants
KTP_UDPGlc    = 0.989957;
KTP_UDPGal    = 7.1464;
KTP_UDPGlcNAc = 5.04905;
KTP_UDPGalNAc = 11.0558;
KTP_GDPMan    = 0.127219;
KTP_GDPFuc    = 0.1;
KTP_CMPNeu5Ac = 503.213;

# Fout,NSD terms (values taken from '' A theoritical estimate for nucleotide...'' del Val et al. 2016)
Nhcp_lipids_UDPGlc = 1.560e-12; #mmolNSD/cell
Nhcp_lipids_UDPGlcNAc = 1.248e-12;
Nhcp_lipids_UDPGal = 2.288e-12;
Nhcp_lipids_UDPGalNAc = 1.252e-12;
Nhcp_lipids_GDPMan = 3.538e-12;
Nhcp_lipids_GDPFuc = 0.140e-12;
Nhcp_lipids_CMPNeu5Ac = 1.846e-12;

NmAb_UDPGlc = 40.39e-6; #mmolNSD/mgmAb        
NmAb_UDPGlcNAc = 26.67e-6; #49.14e-6; #the 49.14 nmol/mg is the total UDPGlcNAc in the mAb while the 26.67 is the only the two molceules of GlcNAc in the core glycan (4 mol_GlcNAc/mol_mAb)
NmAb_UDPGlcNAc_b = 49.14e-6 - 26.67e-6;  
NmAb_UDPGal = 7.119e-6;                        
NmAb_UDPGalNAc = 0.;
NmAb_GDPMan = 121.2e-6;                        
NmAb_GDPFuc = 12.23e-6;
NmAb_CMPNeu5Ac = 0.155e-6;


## 7. State-dependent rate expressions

This section defines the function used to evaluate all state-dependent kinetic quantities in the process model. Given the current state vector, the function computes:

- Specific growth and death rates
- Extracellular uptake and secretion rates
- Intracellular NSD synthesis, conversion, and sink terms
- Uridine- and galactose-dependent auxiliary reaction rates

The function is evaluated at each simulation time step and is used consistently across model simulation and Ensemble Kalman Filter prediction steps. The functional form and ordering of returned quantities are retained from the original implementation to ensure identical numerical behaviour.


In [23]:
# Calculate non-fixed parameters for the process model
# Input: state
# Output: model parameters

def model_params(state, ng):
    
    Xv, mAb, Gal, Urd, Glc, Amm, Gln, Lac, Asn,  Glu, UDPGal, UDPGalNAc, UDPGlc, UDPGlcNAc, GDPMan, GDPFuc, CMPNeu5Ac = state
    
    # Growth and death rate
    flim =Glc/(Kglc+Glc)*Asn/(Kasn+Asn)
    finh = (KIamm/(KIamm + Amm)) * (KIlac/(KIlac + Lac)) * (KIurd/(KIurd + Urd))
    mu = mu_max*flim*finh;
    mu_d = mu_d_max*(Amm/(Kd_amm+Amm)+Urd/(Kd_urd+Urd)) 
    
    # Metabolites
    Qgal=(-mu/Yx_gal)*(Gal/(Gal+Kgal)); #Galactose
    Qurd = -Urd/(Kurd+Urd)*mu/Yx_urd; #Uridine
   
    
    Qglc =(-mu/Yx_glc - m_glc)*(Kc_gal/(Kc_gal+Gal))**ng;  #glucose
    ng = 1 - factor*(Qgal/Qglc);
    
    Qamm= mu/Yx_amm - Yamm_urd*Qurd #ammonia
    Qlac=(mu/Yx_lac  - Ylac_glc*Qglc)*(Lac_max_1 -Lac)/Lac_max_1 + m_lac*(Lac_max_2-Lac)/(Lac_max_2)
    Qglu = -mu/Yx_glu #glutamate
    Qasn = (mu * (Yx_asp-Yx_asn*Yasn_asp))/(Yx_asn*Yx_asp)*(Yasn_asp*Yasp_asn-1) 
    Qgln = mu/Yx_gln - Qglu*Ygln_glu - Qasn*Ygln_asn + Ygln_amm*Qamm  #glutamin
    Gln_int = f2 * Gln #glutamin intracellular
    QmAb = YmAb_mu*mu + YmAb_Xv  #specific productivity
    
    # NSD
    Fout_UDPGal = UDPGal/(KTP_UDPGal + UDPGal)*(Nhcp_lipids_UDPGal*mu/V_cell  + NmAb_UDPGal*QmAb/V_cell)
    Fout_UDPGalNAc = UDPGalNAc/(KTP_UDPGalNAc + UDPGalNAc)*(Nhcp_lipids_UDPGalNAc*mu/V_cell  +  NmAb_UDPGalNAc*QmAb/V_cell)    
    Fout_UDPGlc = UDPGlc/(KTP_UDPGlc + UDPGlc)*(Nhcp_lipids_UDPGlc*mu/V_cell  +  NmAb_UDPGlc*QmAb/V_cell)
    Fout_UDPGlcNAc = UDPGlcNAc/(KTP_UDPGlcNAc + UDPGlcNAc)*(Nhcp_lipids_UDPGlcNAc*mu/V_cell  
                                                            +  NmAb_UDPGlcNAc*QmAb/V_cell + NmAb_UDPGlcNAc_b*QmAb/V_cell)
    Fout_GDPMan = GDPMan/(KTP_GDPMan + GDPMan)*(Nhcp_lipids_GDPMan*mu/V_cell  +  NmAb_GDPMan*QmAb/V_cell)
    Fout_GDPFuc = GDPFuc/(KTP_GDPFuc + GDPFuc)*(Nhcp_lipids_GDPFuc*mu/V_cell  + NmAb_GDPFuc*QmAb/V_cell)
    Fout_CMPNeu5Ac = CMPNeu5Ac/(KTP_CMPNeu5Ac + CMPNeu5Ac)*(Nhcp_lipids_CMPNeu5Ac*mu/V_cell  + NmAb_CMPNeu5Ac*QmAb/V_cell)

    r1_f = Vmax1* Gln_int / (Kdf_E1_gln + Gln_int)
    r2_f  = Vmax2 * Glc / (Kdf_Glc_UDPGlc + Glc)
    r2_bf  = Vmax2b*UDPGal/ (Kdf_E2b*(1 + UDPGlcNAc/KiE2A + UDPGalNAc/KiE2B + UDPGlc/KiE2C + UDPGal/KiE2D) + UDPGal)
    r3_f = Vmax3 * Glc/ (Kdf_Glc_GDPMan + Glc)
    r4_f  = Vmax4 * UDPGlcNAc/ (Kdf_E4 + UDPGlcNAc)
    #r4_bf = Vmax4b * UDPGalNAc / (Kdf_E4 + UDPGalNAc) 
    r5_f  = Vmax5 * UDPGlcNAc/ (UDPGlcNAc + Kdf_E5*(1+CMPNeu5Ac/KiE5))
    
    r6_f  = (Vmax6*UDPGlc) / (Kdf_E6*(1 + UDPGlcNAc/KiE6A + UDPGalNAc/KiE6B + UDPGal/KiE6C) + UDPGlc)
    
    r6_gal =  Vmax6g*Gal/ (K6g*(1+UDPGal/Ki6_ugal + Gal/Ki6_gal + Urd/Ki6_urd) + Gal) 
    r7_f = Vmax7 * GDPMan/ ((Kdf_E7 + GDPMan)*(1+GDPFuc/Ki_E7))
    r7_sink = Vmax7_sink * (GDPFuc/(GDPFuc+K7_sink))
    r1_sink =  Vmax1_sink * UDPGlcNAc / ((UDPGlcNAc+K1_sink)*(1+CMPNeu5Ac/Ki1_sink))
    r6_sink = Vmax6_sink * UDPGal/(UDPGal + K6_sink*(1+UDPGlc/Ki6_glc))*(Gal/(Gal+0.00001))

    # Uridine rates
    r1_urd = Vmax1u*Urd/(K1u + Urd)
    r2_urd = Vmax2u*Urd/(K2u + Urd)
    r4_urd = Vmax4u*Urd/(K4u + Urd)
    r6_urd = Vmax6u*Urd/(K6u + Urd)
    
    return mu, mu_d, Qgal, Qurd, Qglc, Qamm, Qgln, Qlac, Qglu, Qasn, Gln_int, QmAb,\
    Fout_UDPGal, Fout_UDPGalNAc, Fout_UDPGlc, Fout_UDPGlcNAc, Fout_GDPMan, Fout_GDPFuc, Fout_CMPNeu5Ac,\
    r1_f, r2_f, r2_bf, r3_f, r4_f, r5_f, r6_f, r6_gal, r7_f, r7_sink, r1_sink, r6_sink,\
    r1_urd, r2_urd, r4_urd, r6_urd


## 8. Initial conditions

This section defines the procedure used to initialise the model state vector for each experimental dataset. Initial conditions are constructed directly from the experimental measurements by extracting the first available data point from the extracellular metabolite and intracellular NSD dat


In [24]:
# ---------------------------------------------------------------------
# Initial condition construction
# ---------------------------------------------------------------------
# This function extracts initial values from the experimental datasets
# and assembles them into a state vector consistent with `state_name`.

def get_initial_condition(met_df, nsd_df, row_idx=2):
    """
    Construct initial conditions from experimental data.

    Parameters
    ----------
    met_df : pandas.DataFrame
        DataFrame from the 'Metabolites' sheet.
    nsd_df : pandas.DataFrame
        DataFrame from the 'NSD' sheet.
    row_idx : int, optional
        Row index used to extract initial values.
        The default value (2) matches the original scripts.

    Returns
    -------
    initial_condition : dict
        Dictionary mapping state names to initial values.
    state_init : numpy.ndarray
        Initial state vector ordered according to `state_name`.
    """
    initial_condition = {
        # Extracellular and product-related states
        "Xv":  met_df["Xv"][row_idx],
        "mAb": met_df["mAb"][row_idx],
        "Gal": met_df["Gal"][row_idx],
        "Urd": met_df["Urd"][row_idx],
        "Glc": met_df["Glc"][row_idx],
        "Amm": met_df["Amm"][row_idx],
        "Gln": met_df["Gln"][row_idx],
        "Lac": met_df["Lac"][row_idx],
        "Asn": met_df["Asn"][row_idx],

        # Glutamate is fixed at the initial time point,
        # consistent with the original implementation
        "Glu": 2.125,

        # Intracellular NSD states
        "UDPGal":    nsd_df["UDPGal"][row_idx],
        "UDPGalNAc": nsd_df["UDPGalNAc"][row_idx],
        "UDPGlc":    nsd_df["UDPGlc"][row_idx],
        "UDPGlcNAc": nsd_df["UDPGlcNAc"][row_idx],
        "GDPMan":    nsd_df["GDPMan"][row_idx],
        "GDPFuc":    nsd_df["GDPFuc"][row_idx],
        "CMPNeu5Ac": nsd_df["CMPNeu5Ac"][row_idx],
    }

    # Assemble state vector in the predefined order
    state_init = np.array(
        [initial_condition[k] for k in state_name],
        dtype=float
    )

    return initial_condition, state_init


# ---------------------------------------------------------------------
# Build initial conditions for all selected datasets
# ---------------------------------------------------------------------
# Initial conditions are stored both as dictionaries (for readability)
# and as ordered vectors (for numerical simulation).
init_cond_by_dataset = {}
state_init_by_dataset = {}

for name in DATASETS.keys():  # P1–P4
    # Load dataset-specific measurement data
    data = load_dataset(name)
    met_df = data["met_df"]
    nsd_df = data["nsd_df"]

    # Construct initial conditions
    init_cond, state_init = get_initial_condition(
        met_df,
        nsd_df,
        row_idx=2
    )

    init_cond_by_dataset[name] = init_cond
    state_init_by_dataset[name] = state_init


In [26]:


def _to_1d(x):
    return np.asarray(x, dtype=float).reshape(-1)

def _to_scalar(x):
    return np.asarray(x).reshape(-1)[0].item()

def model_step(current_state, time, controls, step_len):
    """
    One integration step of the full mechanistic model.

    controls can be:
      - dict with keys: Fin, Fout, V, Gal_feed, Urd_feed
      - OR list/tuple in order: [Fin, Fout, V, Gal_feed, Urd_feed]
    """
    current_state = _to_1d(current_state)
    step_len = _to_scalar(step_len)

    if isinstance(controls, dict):
        Fin = controls["Fin"]
        Fout = controls["Fout"]
        V = controls["V"]
        Gal_feed = controls["Gal_feed"]
        Urd_feed = controls["Urd_feed"]
    else:
        Fin, Fout, V, Gal_feed, Urd_feed = controls

    Fin = _to_scalar(Fin)
    Fout = _to_scalar(Fout)
    V = _to_scalar(V)
    Gal_feed = _to_scalar(Gal_feed)
    Urd_feed = _to_scalar(Urd_feed)

    def dynamic_model(t, state):
        Xv, mAb, Gal, Urd, Glc, Amm, Gln, Lac, Asn, Glu, \
        UDPGal, UDPGalNAc, UDPGlc, UDPGlcNAc, GDPMan, GDPFuc, CMPNeu5Ac = state

        mu, mu_d, Qgal, Qurd, Qglc, Qamm, Qgln, Qlac, Qglu, Qasn, Gln_int, QmAb, \
        Fout_UDPGal, Fout_UDPGalNAc, Fout_UDPGlc, Fout_UDPGlcNAc, Fout_GDPMan, Fout_GDPFuc, Fout_CMPNeu5Ac, \
        r1_f, r2_f, r2_bf, r3_f, r4_f, r5_f, r6_f, r6_gal, r7_f, r7_sink, r1_sink, r6_sink, \
        r1_urd, r2_urd, r4_urd, r6_urd = model_params(state, ng)

        dXv = Xv * ((mu - mu_d) * V - Fin) / V
        dmAb = (QmAb * V * Xv - Fin * mAb) / V

        dGal = (Fin * (Gal_feed - Gal) + Qgal * V * Xv) / V
        dUrd = (Fin * (Urd_feed - Urd) + Qurd * V * Xv) / V
        dGlc = (Fin * (Glc_feed - Glc) + Qglc * V * Xv) / V
        dAmm = (Fin * (Amm_feed - Amm) + Qamm * V * Xv) / V
        dGln = (Fin * (Gln_feed - Gln) + Qgln * V * Xv) / V
        dLac = (Fin * (Lac_feed - Lac) + Qlac * V * Xv) / V
        dAsn = (Fin * (Asn_feed - Asn) + Qasn * V * Xv) / V
        dGlu = (Fin * (Glu_feed - Glu) + Qglu * V * Xv) / V

        dUDPGal = r6_f + r6_urd + r6_gal - r6_sink - Fout_UDPGal
        dUDPGalNAc = r4_f + r4_urd - Fout_UDPGalNAc
        dUDPGlc = r2_f + r2_bf + r2_urd - Fout_UDPGlc
        dUDPGlcNAc = r1_f + r1_urd - r4_f - r5_f - r1_sink - Fout_UDPGlcNAc
        dGDPMan = r3_f - r7_f - Fout_GDPMan
        dGDPFuc = r7_f - r7_sink - Fout_GDPFuc
        dCMPNeu5Ac = r5_f - Fout_CMPNeu5Ac

        return np.array([
            dXv, dmAb, dGal, dUrd, dGlc, dAmm, dGln, dLac, dAsn, dGlu,
            dUDPGal, dUDPGalNAc, dUDPGlc, dUDPGlcNAc, dGDPMan, dGDPFuc, dCMPNeu5Ac
        ], dtype="float64")

    ode = scp.ode(dynamic_model)
    ode.set_integrator("lsoda", nsteps=3000)
    ode.set_initial_value(current_state, time)

    new_state = ode.integrate(ode.t + step_len)
    return _to_1d(new_state)


In [31]:

simulation_results = {}

for name in tqdm(DATASETS.keys(), desc="Datasets", colour="blue"):
    data = load_dataset(name)
    init_state = state_init_by_dataset[name]

    # Build schedule
    Fin, Fout, Gal_feed, Urd_feed = build_schedule(name)
    V_traj = volume_results_P[name][1:]   # length = steps_n

    state = init_state
    traj = []

    # Inner progress bar for time steps
    for k in tqdm(range(steps_n),
                  desc=f"Simulating {name}",
                  leave=False,
                  colour="green"):
        
        controls_k = {
            "Fin": Fin[k],
            "Fout": Fout[k],
            "V": V_traj[k],
            "Gal_feed": Gal_feed[k],
            "Urd_feed": Urd_feed[k],
        }

        state = model_step(state, time[k], controls_k, step_len[k])
        traj.append(state)

    simulation_results[name] = np.array(traj)

print("✅ All dataset simulations completed!")


Datasets:   0%|          | 0/4 [00:00<?, ?it/s]

Simulating P1:   0%|          | 0/28800 [00:00<?, ?it/s]

Simulating P2:   0%|          | 0/28800 [00:00<?, ?it/s]

Simulating P3:   0%|          | 0/28800 [00:00<?, ?it/s]

Simulating P4:   0%|          | 0/28800 [00:00<?, ?it/s]

✅ All dataset simulations completed!


## 9. Uncertainty specification for Ensemble Kalman Filtering

The Ensemble Kalman Filter (EnKF) requires explicit specification of process and measurement uncertainty to represent model mismatch and experimental noise during state estimation. In this work, uncertainty levels are defined at the level of individual state variables and are kept constant across all four datasets (P1–P4). This ensures that differences in estimation performance reflect the experimental data and feeding strategies rather than dataset-specific re-tuning.

Process uncertainty is specified through `process_noise_var`, which defines the variance of model error assigned to each state. Larger values are used for states that are weakly observed, strongly nonlinear, or expected to exhibit higher biological variability. These variances are assembled into a diagonal matrix $P$ and scaled by a global factor $k_Q$ to obtain the process noise covariance matrix:

$$
Q = k_Q \, P
$$

The scalar \(k_Q\) provides a simple mechanism to adjust overall model flexibility while preserving the relative weighting of uncertainty across states.

Measurement uncertainty is specified through `measurement_noise_var`, which defines the variance of experimental noise for the directly observed extracellular states. Only measurable variables are included, reflecting routine offline analytics. The measurement noise covariance matrix is constructed as:

$$
R = \mathrm{diag}(\sigma_z^2)
$$

where the diagonal entries are taken from `measurement_noise_var`.

The measurement operator \(H\) maps the full 17-dimensional model state onto the measurable subspace. It is constructed as a block matrix composed of an identity block \(A\) for the measurable extracellular states and a zero block \(B\) for the unmeasured intracellular NSDs:

$$
H = [A \; B]
$$

This formulation enforces that NSD states are inferred indirectly through their coupling with the measured extracellular dynamics.

To ensure reproducibility, a fixed random seed is applied prior to ensemble generation so that ensemble realisations and filtering outputs remain consistent across runs.


In [34]:

#process noise variance
process_noise_var = {'Xv': 2.45e+15,
 'mAb': 300,
 'Gal': 1.0,
 'Urd': 1.0,
 'Glc': 20.0,
 'Amm': 0.1625,
 'Gln': 0.4,
 'Lac': 1.16,
 'Asn': 0.01,
 'Glu': 0.048,
 'UDPGal': 0.2,
 'UDPGalNAc':0.01,
 'UDPGlc': 0.06,
 'UDPGlcNAc': 0.01,
 'GDPMan': 0.0002,
 'GDPFuc': 0.0002,
 'CMPNeu5Ac': 0.2}

# measurement noise variance, no Asn or NSD measurements are used in update step therefore no measurement noise is specified.
measurement_noise_var = {
 'Xv': 2.2e+14, #2.2e+14,
 'mAb': 10,
 'Gal': 0.002,
 'Urd': 0.001,
 'Glc': 0.08,
 'Amm': 0.005, 
 'Gln': 1e-4,
 'Lac': 0.4}


In [35]:

ensemble_size = 100

var_model = np.array(list(process_noise_var.values()).copy())
var_meas = np.array(list(measurement_noise_var.values()).copy())
# state conditional covariance
P = np.diag(var_model)

# process noise co-variance matrix
kQ = 2e-6
Q = kQ * P 

# measurement matrix
A = np.identity(meas_num)  #identity matrix for metabolites
B = np.zeros((meas_num, state_num-meas_num)) #unmeasured NSD fill matrix with zeros
H = np.hstack((A, B))

# measurement noise co-variance matrix
R = np.diag(var_meas[:meas_num])

np.random.seed(42)

dt_model = 0.01       # hours
t_total = 288         # hours
N_model = int(t_total / dt_model)
T_model = np.linspace(0, t_total, N_model + 1)


## 10. Multi-dataset preparation and open-loop model simulation

This code block prepares all dataset-specific objects required for **open-loop model simulation** and **Ensemble Kalman Filtering (EnKF)**, and then runs the mechanistic model forward in time without data assimilation. It serves two roles:

1. **Pre-processing for EnKF**  
   (noise realisations, initial states, and measurement timing)
2. **Baseline open-loop simulation**  
   (model behaviour driven only by inputs, without measurement updates)


In [38]:

# ------------------------------------------------------------------
# Multi-dataset preparation of:
# 1) process noise realisations
# 2) model initial state
# 3) measurement timing indices and times
# ------------------------------------------------------------------

# Universal KF timing (same for all datasets)
T_kf = T_model
dt_kf = dt_model
N_kf = len(T_kf) - 1

# Universal measurement interval: 24 h / 0.01 h = 2400 steps
interval = int(24.0 / dt_model)  # 2400

# If all datasets share identical measurement times, keep this fixed
T_meas_fixed = np.array([
    0., 24.01, 48.01, 72.01, 96., 96.03, 120.01, 144.01, 144.03,
    168.01, 192.01, 192.03, 216.01, 240.01, 240.03, 264.01, 288.0
])

# Store per-dataset objects here
noise_model_by_dataset = {}
state_model_by_dataset = {}
set_model_by_dataset = {}
T_index_meas_by_dataset = {}
T_meas_by_dataset = {}
N_meas_time_by_dataset = {}

# Reproducible but distinct noise per dataset
seed_base = 42

for name in DATASETS.keys():  # P1–P4
    data = load_dataset(name)
    set_meas = data["set_meas"]   # dataset-specific measurements

    # -------- process noise (dataset-specific realisation) --------
    # Use a different seed for each dataset but reproducible overall
    try:
        ds_id = int(name[1:])   # "P1" -> 1, ...
    except:
        ds_id = 0

    rng = np.random.default_rng(seed_base + ds_id)

    noise_model = rng.multivariate_normal(
        mean=np.zeros(state_num),
        cov=np.diag(var_model),
        size=N_model
    )

    noise_model_by_dataset[name] = noise_model

    # -------- initial model state (dataset-specific) --------
    state_model = state_init_by_dataset[name].copy()
    state_model_by_dataset[name] = state_model

    # store simulated trajectory list initialised at t=0
    set_model_by_dataset[name] = [state_model.copy()]

    # -------- measurement timing (dataset-specific length) --------
    N_meas_time = set_meas.shape[0]
    N_meas_time_by_dataset[name] = N_meas_time

    T_index_meas = [i * interval for i in range(N_meas_time)]
    T_index_meas_by_dataset[name] = T_index_meas

    # If you trust all datasets share the same sampling pattern, use fixed times.
    # Otherwise derive from indices to be fully robust:
    if len(T_meas_fixed) == N_meas_time:
        T_meas = T_meas_fixed.copy()
    else:
        T_meas = T_model[T_index_meas]

    T_meas_by_dataset[name] = T_meas

print("Prepared noise, initial states, and measurement timing for all datasets.")


Prepared noise, initial states, and measurement timing for all datasets.


In [39]:

# ------------------------------------------------------------------
# Multi-dataset open-loop simulation (model only)
# Outputs:
#   set_model_by_dataset[name] : (steps_n+1, state_num) array
#   t_list_by_dataset[name]    : time vector aligned to that trajectory
# ------------------------------------------------------------------

set_model_by_dataset = {}
t_list_by_dataset = {}

for name in tqdm(DATASETS.keys(), desc="Datasets"):
    # dataset-specific initial condition and schedules
    init_cond = init_cond_by_dataset[name]              # dict from get_initial_condition
    current_state = state_init_by_dataset[name].copy()  # 17-vector ordered as state_name

    Fin, Fout, Gal_feed, Urd_feed = build_schedule(name)
    V_traj = volume_results_P[name][1:]  # length = steps_n, aligned to step grid

    # storage: start with initial state at t = 0
    traj = [current_state.copy()]
    t_list = [0.0]

    for k in tqdm(range(steps_n), desc=f"Simulating {name}", leave=False):
        controls_k = {
            "Fin": Fin[k],
            "Fout": Fout[k],
            "V": V_traj[k],
            "Gal_feed": Gal_feed[k],
            "Urd_feed": Urd_feed[k],
        }

        current_state = model_step(current_state, 0.0, controls_k, step_len[k])
        traj.append(current_state.copy())
        t_list.append(t_list[-1] + step_len[k])

    set_model_by_dataset[name] = np.vstack(traj)   # shape (steps_n+1, state_num)
    t_list_by_dataset[name] = np.array(t_list)

# Example access
set_model_by_dataset["P1"].shape, t_list_by_dataset["P1"][:5]


Datasets:   0%|          | 0/4 [00:00<?, ?it/s]

Simulating P1:   0%|          | 0/28800 [00:00<?, ?it/s]

Simulating P2:   0%|          | 0/28800 [00:00<?, ?it/s]

Simulating P3:   0%|          | 0/28800 [00:00<?, ?it/s]

Simulating P4:   0%|          | 0/28800 [00:00<?, ?it/s]

((28801, 17), array([0.  , 0.01, 0.02, 0.03, 0.04]))

## 11. Generating Measurement Ensembles

The Ensemble Kalman Filter requires multiple noisy realisations of each experimental measurement rather than using a single measured value.  
To achieve this, the measured data for each dataset (P1–P4) are expanded into an ensemble.

For every measurement time point, we take the recorded measurement vector and add small, randomly generated noise based on the measurement noise variances defined earlier.  
This creates many slightly different versions of the same measurement, each representing a plausible observation given the noise in the experiment.

Each dataset therefore produces an array of shape:

- **number of measurement times × ensemble size × number of measured variables**

All noisy values are also clipped to remain non-negative to maintain biological realism.  
These perturbed measurements are then used during the EnKF update step to ensure the filter captures measurement uncertainty appropriately across all four datasets.


In [40]:
set_meas_ens_by_dataset = {}

for name in tqdm(DATASETS.keys(), desc="Generating measurement ensembles"):

    data = load_dataset(name)
    set_meas_full = data["set_meas"]

    N_meas_time, meas_num_full = set_meas_full.shape

    # EnKF uses only Xv..Lac
    set_meas_kf = set_meas_full[:, :meas_num]
    set_meas_ens = np.zeros((N_meas_time, ensemble_size, meas_num))

    # precompute standard deviations
    meas_std = np.sqrt(var_meas[:meas_num])

    for i in range(N_meas_time):
        noise_samples = np.random.multivariate_normal(
            mean=np.zeros(meas_num),
            cov=np.diag(var_meas[:meas_num]),
            size=ensemble_size
        )
        #np.random.seed(42)
        # ---- 3-sigma capping per measurement ----
        for j in range(meas_num):
            noise_samples[:, j] = np.clip(
                noise_samples[:, j],
                -3.0 * meas_std[j],
                3.0 * meas_std[j]
            )

        set_meas_ens[i] = np.clip(
            set_meas_kf[i] + noise_samples,
            a_min=1e-12,
            a_max=None
        )

    set_meas_ens_by_dataset[name] = set_meas_ens

print("✅ Measurement ensembles generated for:", list(set_meas_ens_by_dataset.keys()))


Generating measurement ensembles:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Measurement ensembles generated for: ['P1', 'P2', 'P3', 'P4']


## 12. Ensemble Kalman Filter class

This `EnsembleKalmanFilter` class is universal and does not need to be changed for a multi dataset workflow.  
It defines the core EnKF operations in a dataset agnostic way, including ensemble generation, prediction through the process model, and measurement update using Kalman gain.

Dataset specificity is handled outside the class. For each dataset, you simply create a new filter instance and provide the dataset dependent elements:

- the initial state mean `self.x` and initial covariance passed into `create_ensemble`
- the process noise covariance `self.Q`
- the measurement noise covariance `self.R`
- the measurement operator `self.H`
- the model step function `self.fx`
- the time step `self.dt`
- the measurement ensemble `z_ensemble` at each update time

As long as all datasets share the same state definition and the same set of measurable variables, the class can be reused directly for P1 to P4.  
In practice you will instantiate and run one filter per dataset inside your dataset loop, while keeping this class unchanged.


In [ ]:
class EnsembleKalmanFilter():
    
    def __init__(self, num_x, num_z):
        
        self.x = None  #state mean 
        self.z = None
        self.Q = None
        self.R = None
        self.fx = None
        self.H = None
        self.dt = None 
        self.P = None
        
        self.X = None #ensemble 
        self.num_x = num_x  #number of states
        self.num_X = None # number of ensemble points
        self.num_z = num_z #number of memorable states (metabolites)
        
    def create_ensemble(self, N, Cov):
        self.num_X = N
        self.num_x = len(self.x)
    
        # draw initial ensemble
        self.X = np.random.multivariate_normal(self.x, Cov, N)
    
        # ---- cap initial ensemble spread per state (3-sigma rule) ----
        for i in range(self.num_x):
            sd = np.sqrt(Cov[i, i])
            self.X[:, i] = np.clip(self.X[:, i], self.x[i] - 3.0 * sd, self.x[i] + 3.0 * sd)
    
        # enforce positivity
        self.X = np.clip(self.X, a_min=1e-12, a_max=None)
    
        # set ensemble mean
        self.x = np.mean(self.X, axis=0)

    
    def predict(self, controls):
        X = []
        for x in self.X:
            X.append(self.fx(x, 0.0, controls, self.dt))
        #np.random.seed(42)
         # draw process noise
        noise = np.random.multivariate_normal(mean=np.zeros(self.num_x), cov=self.Q, size=self.num_X)
        
        # ---- cap noise per state (3-sigma rule) ----
        for i in range(self.num_x):
            sd = np.sqrt(self.Q[i, i])
            noise[:, i] = np.clip(noise[:, i], -3.0 * sd, 3.0 * sd)
            
        self.X = np.array(X) + noise
        
        self.X = np.clip(self.X, a_min=1e-12, a_max=None)  # Ensures all values are >= 0
        self.x = np.mean(self.X, axis=0)
        
    def update(self, z_ensemble):
        N = self.num_X  # should match z_ensemble.shape[0]
    
        # Compute state mean and anomalies
        x_mean = np.mean(self.X, axis=0)
        E_x = self.X - x_mean  # shape: (N, num_x)

        # Compute measurement mean and anomalies
        Z = np.array([self.H @ x for x in self.X])  # predicted measurements from states
        z_mean = np.mean(Z, axis=0)
        E_z = Z - z_mean  # shape: (N, num_z)

        # Covariances
        P_xz = (E_x.T @ E_z) / (N - 1)  # cross-covariance
        P_zz = (E_z.T @ E_z) / (N - 1) + self.R  # measurement covariance + noise
    
        # Kalman gain
        K = P_xz @ inv(P_zz)
    
        # Measurement update
        self.X += (K @ (z_ensemble - Z).T).T  # shape: (N, num_x)
        self.X = np.clip(self.X, a_min=1e-12, a_max=None)
        self.x = np.mean(self.X, axis=0)

In [46]:
enkf_results_by_dataset = {}
enkf_runs_by_dataset = {}

rmse_records = []


decimal_places = 2
n_runs = 10

In [ ]:


for name in tqdm(DATASETS.keys(), desc="Running EnKF over datasets"):
    print(f"\nRunning EnKF for {name}")

    # -------- Dataset specific data --------
    data = load_dataset(name)

    # Full measurements from file (includes Asn as last column)
    set_meas_full = data["set_meas"]                     # shape (N_meas_time, meas_num_full)

    # 👉 EnKF uses only first meas_num variables (Xv..Lac), drop Asn
    set_meas = set_meas_full[:, :meas_num].copy()        # shape (N_meas_time, meas_num)

    set_meas_ens = set_meas_ens_by_dataset[name]         # (N_meas_time, ensemble_size, meas_num)
    T_meas = T_meas_by_dataset[name]                     # (N_meas_time,)

    # Controls and volume
    Fin, Fout, Gal_feed, Urd_feed = build_schedule(name)
    V_traj = volume_results_P[name][1:]                  # length = steps_n

    # Initial state
    state_init = state_init_by_dataset[name].copy()

    # Time alignment
    time_steps_A = [round(i * dt_kf, decimal_places) for i in range(N_kf)]
    time_steps_B = [round(t, decimal_places) for t in T_meas.tolist()]
    meas_time_to_index = {t: i for i, t in enumerate(time_steps_B)}

    list_EnKF = []

    for run_i in range(n_runs):
        EnKF = EnsembleKalmanFilter(state_num, meas_num)
        EnKF.x = state_init.copy()
        EnKF.z = set_meas.copy()     # (N_meas_time, meas_num) = Xv..Lac only
        EnKF.Q = Q.copy()
        EnKF.R = R.copy()
        EnKF.H = H.copy()
        EnKF.fx = model_step
        EnKF.dt = dt_kf

        # Create initial ensemble around initial state
        EnKF.create_ensemble(ensemble_size, Q)

        set_EnKF = [state_init.copy()]

        for idx_A, step_A in enumerate(
            tqdm(time_steps_A, desc=f"{name} run {run_i+1}/{n_runs}", leave=False)
        ):
            controls_k = {
                "Fin": Fin[idx_A],
                "Fout": Fout[idx_A],
                "V": V_traj[idx_A],
                "Gal_feed": Gal_feed[idx_A],
                "Urd_feed": Urd_feed[idx_A],
            }
            EnKF.predict(controls_k)

            # Update step if a measurement exists at this time
            if step_A in meas_time_to_index:
                idx_B = meas_time_to_index[step_A]
                z_ens = set_meas_ens[idx_B]   # (ensemble_size, meas_num), no Asn
                EnKF.update(z_ens)

            set_EnKF.append(EnKF.x.copy())

        list_EnKF.append(np.array(set_EnKF))

    set_EnKF_mean = np.mean(list_EnKF, axis=0)
    
    enkf_results_by_dataset[name] = set_EnKF_mean
    enkf_runs_by_dataset[name] = list_EnKF

print("All datasets processed with EnKF.")


In [ ]:


for name in DATASETS.keys():
    # ---------------- Dataset-specific data ----------------
    data = load_dataset(name)

    # Metabolite measurements (all measured metabolites, e.g. Xv..Asn)
    set_meas = data["set_meas"].astype(float)          # shape (N_meas_time, n_met)
    n_met = set_meas.shape[1]

    # NSD measurements
    NSD_meas = (
        pd.DataFrame(data["NSD_meas"])
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy()
    )                                                  # shape (N_meas_time, n_nsd)
    n_nsd = NSD_meas.shape[1]

    T_meas = T_meas_by_dataset[name]

    # model, EnKF and EKF trajectories
    set_model = set_model_by_dataset[name]             # (N_model+1, state_num)
    set_EnKF  = enkf_results_by_dataset[name]          # (N_kf+1, state_num)
    
    # ---------------- Metabolites: global state indices 0 .. n_met-1 ----------------
    for i in range(n_met):
        measured   = set_meas[:, i].astype(float)
        model_pred = np.interp(T_meas, T_model, set_model[:, i])
        enkf_pred  = np.interp(T_meas, T_kf,    set_EnKF[:, i])
       

        # mask valid measurements
        mask = ~np.isnan(measured)
        if np.sum(mask) == 0:
            continue

        measured_values = measured[mask]
        model_values    = model_pred[mask]
        enkf_values     = enkf_pred[mask]
       

        model_rmse = np.sqrt(mean_squared_error(measured_values, model_values))
        enkf_rmse  = np.sqrt(mean_squared_error(measured_values, enkf_values))
        
        rmse_records.append({
            "Dataset":    name,
            "State":      axis_name[i],   # e.g. Xv, mAb, Gal, ..., Asn
            "RMSE_Model": model_rmse,
            "RMSE_EnKF":  enkf_rmse
        })

    # ---------------- NSDs: assume they occupy the last n_nsd states ----------------
    # e.g. state indices 10..16 when state_num=17 and n_nsd=7
    start_nsd = state_num - n_nsd

    for j in range(n_nsd):
        i = start_nsd + j   # global state index of NSD j

        measured   = NSD_meas[:, j].astype(float)
        model_pred = np.interp(T_meas, T_model, set_model[:, i])
        enkf_pred  = np.interp(T_meas, T_kf,    set_EnKF[:, i])
    
        mask = ~np.isnan(measured)
        if np.sum(mask) == 0:
            continue

        measured_values = measured[mask]
        model_values    = model_pred[mask]
        enkf_values     = enkf_pred[mask]

        model_rmse = np.sqrt(mean_squared_error(measured_values, model_values))
        enkf_rmse  = np.sqrt(mean_squared_error(measured_values, enkf_values))
        
        rmse_records.append({
            "Dataset":    name,
            "State":      axis_name[i],   # NSD axis label
            "RMSE_Model": model_rmse,
            "RMSE_EnKF":  enkf_rmse
        })

# ---------------- Build final table ----------------
rmse_df_all = pd.DataFrame(rmse_records)
rmse_df_all = rmse_df_all.round(3)



In [ ]:
# Figure output subdirectory (under BASE_RESULTS_DIR)
folder_name = "figures"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

# ----------------------------
# NSD indices
# ----------------------------
nsd_states_global = {
    "UDPGal": 10,
    "UDPGalNAc": 11,
    "UDPGlc": 12,
    "UDPGlcNAc": 13,
    "GDPMan": 14,
    "GDPFuc": 15,
    "CMPNeu5Ac": 16
}

nsd_states_local = {
    "UDPGal": 0,
    "UDPGalNAc": 1,
    "UDPGlc": 2,
    "UDPGlcNAc": 3,
    "GDPMan": 4,
    "GDPFuc": 5,
    "CMPNeu5Ac": 6
}

state_order = ["UDPGal","UDPGalNAc","UDPGlc","UDPGlcNAc","GDPMan","GDPFuc","CMPNeu5Ac"]
dataset_list = list(DATASETS.keys())  # P1..P4

# ----------------------------
# Styling for individual runs
# ----------------------------
plot_individual_runs = True
max_runs_to_plot = None        # e.g. 10, or None for all
run_lw = 1.0
run_alpha = 0.18
mean_lw = 2.4

n_rows = len(state_order)
n_cols = len(dataset_list)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(4.8 * n_cols, 3.9 * n_rows),
    sharex=True
)

if n_rows == 1:
    axes = np.expand_dims(axes, 0)
if n_cols == 1:
    axes = np.expand_dims(axes, 1)

for r, state_key in enumerate(state_order):
    i_global = nsd_states_global[state_key]
    j_local  = nsd_states_local[state_key]

    for c, name in enumerate(dataset_list):
        ax = axes[r, c]

        # trajectories: model + EnKF mean
        set_model = set_model_by_dataset[name]
        set_EnKF_mean = enkf_results_by_dataset[name]

        # individual runs (list of arrays shape (N_kf+1, state_num))
        list_EnKF = enkf_runs_by_dataset.get(name, None)

        # measurements
        data = load_dataset(name)
        NSD_meas = data["NSD_meas"]
        NSD_meas_errorbar = data["NSD_meas_errorbar"]
        T_meas = T_meas_by_dataset[name]

        # model
        ax.plot(T_model, set_model[:, i_global], color="red", lw=2.2)

        # individual EnKF runs (thin, transparent)
        if plot_individual_runs and list_EnKF is not None and len(list_EnKF) > 0:
            runs_to_plot = list_EnKF if max_runs_to_plot is None else list_EnKF[:max_runs_to_plot]
            for run_arr in runs_to_plot:
                ax.plot(T_kf, run_arr[:, i_global], color="black", lw=run_lw, alpha=run_alpha)

        # EnKF mean (thicker dashed)
        ax.plot(T_kf, set_EnKF_mean[:, i_global], color="black", ls="--", lw=mean_lw)

        # measurements
        ax.errorbar(
            T_meas,
            NSD_meas[:, j_local],
            yerr=NSD_meas_errorbar[:, j_local],
            fmt="o",
            color="orange",
            markersize=4.5,
            capsize=2,
            elinewidth=1,
            alpha=0.9
        )

        # titles & labels
        if r == 0:
            ax.set_title(name, fontsize=13, fontweight="bold")
        if c == 0:
            ax.set_ylabel(axis_name[i_global], fontsize=11, fontweight="bold")
        if r == n_rows - 1:
            ax.set_xlabel("Time (hours)", fontsize=11, fontweight="bold")

        ax.grid(alpha=0.12)
        ax.tick_params(axis="both", labelsize=10)

        # bold tick labels
        for lab in ax.get_xticklabels() + ax.get_yticklabels():
            lab.set_fontweight("bold")

        # thicker spines
        for spine in ax.spines.values():
            spine.set_linewidth(2)

# ----------------------------
# Shared legend
# ----------------------------
legend_elements = [
    Line2D([0], [0], color="red", lw=2.3, label="Mechanistic Model"),
    Line2D([0], [0], color="black", lw=run_lw, alpha=0.45, label="EnKF individual runs"),
    Line2D([0], [0], color="black", lw=mean_lw, ls="--", label="EnKF mean"),
    Line2D([0], [0], color="orange", marker="o", lw=0, markersize=6, label="Measurements"),
]

fig.legend(
    handles=legend_elements,
    loc="lower center",
    ncol=4,
    fontsize=12,
    frameon=False,
    bbox_to_anchor=(0.5, -0.01)
)

# ----------------------------
# Save
# ----------------------------
save_dir = BASE_RESULTS_DIR / folder_name
save_dir.mkdir(parents=True, exist_ok=True)
fig_name = "all_nsd_with_individual_runs.png"

plt.tight_layout(rect=[0, 0.05, 1, 1])
#plt.savefig(save_dir / fig_name, dpi=600, bbox_inches="tight")
plt.show()

#print("✅ Figure saved to:", save_dir / fig_name)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

# --- NSDs to plot (global indices in state vector) ---
nsd_states_global = {
    "UDPGal": 10,
    "UDPGalNAc": 11,
    "UDPGlc": 12,
    "UDPGlcNAc": 13,
    "GDPMan": 14,
    "GDPFuc": 15,
    "CMPNeu5Ac" :16
}

# NSD_meas column indices
nsd_states_local = {
    "UDPGal": 0,
    "UDPGalNAc": 1,
    "UDPGlc": 2,
    "UDPGlcNAc": 3,
    "GDPMan": 4,
    "GDPFuc": 5,
    "CMPNeu5Ac" :6
}

state_order = ["UDPGal","UDPGalNAc", "UDPGlc", "UDPGlcNAc",  "GDPMan", "GDPFuc","CMPNeu5Ac" ]

dataset_list = list(DATASETS.keys())    # expects 4 datasets

n_rows = len(state_order)   # 3 NSDs
n_cols = len(dataset_list)  # 4 datasets

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(4.5 * n_cols, 4 * n_rows),
    sharex=True
)

# ensure 2D array even if shape collapses
if n_rows == 1:
    axes = np.expand_dims(axes, 0)
if n_cols == 1:
    axes = np.expand_dims(axes, 1)

for r, state_key in enumerate(state_order):
    i_global = nsd_states_global[state_key]
    j_local  = nsd_states_local[state_key]

    for c, name in enumerate(dataset_list):
        ax = axes[r, c]

        # trajectories
        set_model = set_model_by_dataset[name]
        set_EnKF  = enkf_results_by_dataset[name]

        # measurements
        data = load_dataset(name)
        NSD_meas = data["NSD_meas"]
        NSD_meas_errorbar = data["NSD_meas_errorbar"]
        T_meas = T_meas_by_dataset[name]

        # --- Plot curves ---
        ax.plot(T_model, set_model[:, i_global], color="red", lw=2.2, label="Model")
        ax.plot(T_kf, set_EnKF[:, i_global], color="black", ls="--", lw=2.2, label="EnKF")

        ax.errorbar(
            T_meas,
            NSD_meas[:, j_local],
            yerr=NSD_meas_errorbar[:, j_local],
            fmt='o',
            color='orange',
            markersize=4.5,
            capsize=2,
            elinewidth=1,
            alpha=0.9,
            label="Measurement"
        )

        # --- Titles & labels ---
        if r == 0:
            ax.set_title(name, fontsize=13, fontweight='bold')

        if c == 0:
            ax.set_ylabel(axis_name[i_global], fontsize=11)

        if r == n_rows - 1:
            ax.set_xlabel("Time (hours)", fontsize=11)

        ax.grid(alpha=0.12)
        ax.tick_params(axis='both', labelsize=10)

# ---- Shared Legend ----
legend_elements = [
    Line2D([0], [0], color="red", lw=2.3, label="Mechanistic Model"),
    Line2D([0], [0], color="black", lw=2.3, ls="--", label="EnKF"),
    Line2D([0], [0], color="orange", marker='o', lw=0, markersize=6, label="Measurements")
]

fig.legend(
    handles=legend_elements,
    loc="lower center",
    ncol=3,
    fontsize=12,
    frameon=False,
    bbox_to_anchor=(0.5, -0.015)
)



save_dir = BASE_RESULTS_DIR / folder_name
save_dir.mkdir(parents=True, exist_ok=True)


fig_name = "all_nsd.png"

plt.tight_layout(rect=[0, 0.05, 1, 1])
#plt.savefig(save_dir / fig_name,dpi=600,bbox_inches="tight")
plt.show()

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

# --------- Fixed dataset order ----------
dataset_list = ["P1", "P2", "P3", "P4"]   # change if your dataset keys differ
#dataset_list = ["P1"] 
# --------- Colour + marker per dataset ----------
dataset_colours = {
    "P1": "tab:orange",
    "P2": "tab:green",
    "P3": "tab:blue",
    "P4": "purple",
}
dataset_markers = {
    "P1": "o",
    "P2": "s",
    "P3": "^",
    "P4": "D",
}

# --------- NSDs to plot ----------
nsd_states_global = {"UDPGal": 10,  "UDPGlc": 12, "UDPGlcNAc": 13}
nsd_states_local  = {"UDPGal": 0,  "UDPGlc": 2,  "UDPGlcNAc": 3}
state_order = ["UDPGal", "UDPGlc", "UDPGlcNAc"]

n_rows = len(state_order)
n_cols = len(dataset_list)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(4.6 * n_cols, 4.0 * n_rows),
    sharex=False
)

if n_rows == 1:
    axes = np.expand_dims(axes, 0)
if n_cols == 1:
    axes = np.expand_dims(axes, 1)

letters = [f"({chr(97+i)})" for i in range(n_rows * n_cols)]

for r, state_key in enumerate(state_order):
    i_global = nsd_states_global[state_key]
    j_local  = nsd_states_local[state_key]

    for c, name in enumerate(dataset_list):
        ax = axes[r, c]

        colour = dataset_colours[name]
        marker = dataset_markers[name]

        set_model = set_model_by_dataset[name]
        set_EnKF  = enkf_results_by_dataset[name]

        data = load_dataset(name)
        NSD_meas = np.asarray(data["NSD_meas"], dtype=float)
        NSD_err  = np.asarray(data.get("NSD_meas_errorbar", data.get("NSD_meas_error")), dtype=float)
        T_meas   = np.asarray(T_meas_by_dataset[name], dtype=float)

        ax.set_title(letters[r*n_cols + c], fontsize=16, fontweight="bold", loc="left")
        # Optional: dataset name in the top-right (comment out if you don't want it)
        ax.text(0.05, 0.98, name, transform=ax.transAxes,
            ha="left", va="top", fontsize=14, fontweight="bold")

        # Model (solid), EnKF (dashed)
        ax.plot(T_model, set_model[:, i_global], color=colour, lw=2.6, ls="-")
        ax.plot(T_kf,    set_EnKF[:, i_global],  color=colour, lw=2.6, ls="--")

        # Measurements (marker in same colour)
        ax.errorbar(
            T_meas,
            NSD_meas[:, j_local],
            yerr=NSD_err[:, j_local] if NSD_err is not None else None,
            fmt=marker,
            color=colour,
            ecolor="black",
            elinewidth=1.2,
            capsize=2.5,
            markersize=6.0,
            markeredgecolor="black",
            markeredgewidth=0.8,
            alpha=0.95
        )

        ax.set_xlabel("Time (hours)", fontsize=13, fontweight="bold")
        ax.set_ylabel(axis_name[i_global], fontsize=13, fontweight="bold")

        ax.tick_params(axis="both", which="both", labelsize=12, width=1.8, length=4, direction="in")
        for lab in ax.get_xticklabels() + ax.get_yticklabels():
            lab.set_fontweight("bold")

        for spine in ax.spines.values():
            spine.set_linewidth(2.0)

        ax.grid(alpha=0.12)

# -------------------------
# Legend: Matplotlib fills COLUMN-first.
# So provide handles grouped by dataset to make each dataset one column:
# [P1 Model, P1 EnKF, P1 Meas, P2 Model, P2 EnKF, P2 Meas, ...]
# -------------------------
handles = []
for name in dataset_list:
    c = dataset_colours[name]
    m = dataset_markers[name]

    handles.append(Line2D([0], [0], color=c, lw=2.8, ls="-",  label=f"{name} – Model"))
    handles.append(Line2D([0], [0], color=c, lw=2.8, ls="--", label=f"{name} – EnKF"))
    handles.append(Line2D([0], [0], marker=m, color=c, lw=0,
                          markersize=8, markeredgecolor="black", markeredgewidth=1.0,
                          label=f"{name} – Validation Measurement"))

fig.legend(
    handles=handles,
    ncol=len(dataset_list),          # 4 columns = P1..P4
    loc="lower center",
    bbox_to_anchor=(0.5, -0.03),
    frameon=False,
    fontsize=12.5,
    prop={"weight": "bold"},
    handlelength=2.6,
    handletextpad=0.8,
    columnspacing=2.2,
    labelspacing=0.8   # vertical spacing within each column
)



save_dir = BASE_RESULTS_DIR / folder_name
save_dir.mkdir(parents=True, exist_ok=True)


fig_name = "main_nsd.png"

plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(save_dir / fig_name,dpi=600,bbox_inches="tight")
plt.show()

print("✅ Figure saved to:", save_dir / fig_name)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
import seaborn as sns

sns.set(style="white", context="talk")

# -----------------------------
# What to plot (rows)
# -----------------------------
selected_states = {
    "Xv": 0,
    "Gal": 2,
    "Urd": 3,
    "Glc": 4,
    "Gln": 6
}
state_order = list(selected_states.keys())

# -----------------------------
# Dataset order (columns)
# -----------------------------
dataset_list = list(DATASETS.keys())   # expects 4 datasets, order as in DATASETS

# -----------------------------
# Colour + marker per dataset
# -----------------------------
default_markers = ["o", "s", "^", "D", "X", "P", "v", "<", ">", "*"]
default_colours = ["tab:orange", "tab:green", "tab:blue", "purple", "tab:red", "tab:brown"]

dataset_colours = {name: default_colours[i % len(default_colours)] for i, name in enumerate(dataset_list)}
dataset_markers = {name: default_markers[i % len(default_markers)] for i, name in enumerate(dataset_list)}

# -----------------------------
# Figure layout
# -----------------------------
n_rows = len(state_order)
n_cols = len(dataset_list)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(4.6 * n_cols, 3.8 * n_rows),
    sharex=False
)

if n_rows == 1:
    axes = np.expand_dims(axes, 0)
if n_cols == 1:
    axes = np.expand_dims(axes, 1)

# subplot labels (a), (b), (c)...
letters = [f"({chr(97+i)})" for i in range(n_rows * n_cols)]

# -----------------------------
# Plotting
# -----------------------------
for r, state_key in enumerate(state_order):
    i_global = selected_states[state_key]

    for c, name in enumerate(dataset_list):
        ax = axes[r, c]

        colour = dataset_colours[name]
        marker = dataset_markers[name]

        set_model = set_model_by_dataset[name]
        set_EnKF  = enkf_results_by_dataset[name]

        data = load_dataset(name)
        set_meas = np.asarray(data["set_meas"], dtype=float)

        set_meas_err = data.get("set_meas_errorbar", data.get("set_meas_error"))
        set_meas_err = None if set_meas_err is None else np.asarray(set_meas_err, dtype=float)

        T_meas = np.asarray(T_meas_by_dataset[name], dtype=float)

        # (a),(b)... label inside each axis
        ax.set_title(letters[r*n_cols + c], fontsize=16, fontweight="bold", loc="left")
        # Optional: dataset name in the top-right (comment out if you don't want it)
        ax.text(0.05, 0.98, name, transform=ax.transAxes,
            ha="left", va="top", fontsize=14, fontweight="bold")

        # Model (solid) + EnKF (dashed): same dataset colour
        ax.plot(T_model, set_model[:, i_global], color=colour, lw=2.6, ls="-")
        ax.plot(T_kf,    set_EnKF[:, i_global],  color=colour, lw=2.6, ls="--")

        # Measurements: dataset marker + same dataset colour
        ax.errorbar(
            T_meas,
            set_meas[:, i_global],
            yerr=set_meas_err[:, i_global] if set_meas_err is not None else None,
            fmt=marker,
            color=colour,
            ecolor="black",
            elinewidth=1.2,
            capsize=2.5,
            markersize=6.0,
            markeredgecolor="black",
            markeredgewidth=0.8,
            alpha=0.95
        )

        # Axis labels (individual, bold, bigger)
        ax.set_xlabel("Time (hours)", fontsize=13, fontweight="bold")
        ax.set_ylabel(axis_name[i_global], fontsize=13, fontweight="bold")

        ax.grid(alpha=0.12)

        # Bigger/bold ticks
        ax.tick_params(axis="both", which="both", labelsize=12, width=1.8, length=4, direction="in")
        for lab in ax.get_xticklabels() + ax.get_yticklabels():
            lab.set_fontweight("bold")

        for spine in ax.spines.values():
            spine.set_linewidth(2.0)

# -----------------------------
# Legend: dataset-per-column (P1 column, P2 column, ...)
# Matplotlib fills column-first, so supply handles grouped by dataset:
# [D1 Model, D1 EnKF, D1 Meas, D2 Model, D2 EnKF, D2 Meas, ...]
# -----------------------------
handles = []
for name in dataset_list:
    c = dataset_colours[name]
    m = dataset_markers[name]
    handles.append(Line2D([0], [0], color=c, lw=2.8, ls="-",  label=f"{name} – Model"))
    handles.append(Line2D([0], [0], color=c, lw=2.8, ls="--", label=f"{name} – EnKF"))
    handles.append(Line2D([0], [0], marker=m, color=c, lw=0,
                          markersize=8, markeredgecolor="black", markeredgewidth=1.0,
                          label=f"{name} – Measurement Update"))

fig.legend(
    handles=handles,
    ncol=len(dataset_list),          # one dataset per column
    loc="lower center",
    bbox_to_anchor=(0.5, 0.0),
    frameon=False,
    fontsize=12.5,
    prop={"weight": "bold"},
    handlelength=2.6,
    handletextpad=0.8,
    columnspacing=2.2,
    labelspacing=0.8
)


save_dir = BASE_RESULTS_DIR / folder_name
save_dir.mkdir(parents=True, exist_ok=True)


fig_name = "measured_metabolites.png"

plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(save_dir / fig_name, dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
import seaborn as sns

sns.set(style="white", context="talk")

# -----------------------------
# What to plot (rows)
# -----------------------------
selected_states = {
    "Asn": 8
}
state_order = list(selected_states.keys())

# -----------------------------
# Dataset order (columns)
# -----------------------------
dataset_list = list(DATASETS.keys())   # expects 4 datasets, order as in DATASETS

# -----------------------------
# Colour + marker per dataset
# -----------------------------
default_markers = ["o", "s", "^", "D", "X", "P", "v", "<", ">", "*"]
default_colours = ["tab:orange", "tab:green", "tab:blue", "purple", "tab:red", "tab:brown"]

dataset_colours = {name: default_colours[i % len(default_colours)] for i, name in enumerate(dataset_list)}
dataset_markers = {name: default_markers[i % len(default_markers)] for i, name in enumerate(dataset_list)}

# -----------------------------
# Figure layout
# -----------------------------
n_rows = len(state_order)
n_cols = len(dataset_list)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(6.6 * n_cols, 4.8 * n_rows),
    sharex=False
)

if n_rows == 1:
    axes = np.expand_dims(axes, 0)
if n_cols == 1:
    axes = np.expand_dims(axes, 1)

# subplot labels (a), (b), (c)...
letters = [f"({chr(97+i)})" for i in range(n_rows * n_cols)]

# -----------------------------
# Plotting
# -----------------------------
for r, state_key in enumerate(state_order):
    i_global = selected_states[state_key]

    for c, name in enumerate(dataset_list):
        ax = axes[r, c]

        colour = dataset_colours[name]
        marker = dataset_markers[name]

        set_model = set_model_by_dataset[name]
        set_EnKF  = enkf_results_by_dataset[name]

        data = load_dataset(name)
        set_meas = np.asarray(data["set_meas"], dtype=float)

        set_meas_err = data.get("set_meas_errorbar", data.get("set_meas_error"))
        set_meas_err = None if set_meas_err is None else np.asarray(set_meas_err, dtype=float)

        T_meas = np.asarray(T_meas_by_dataset[name], dtype=float)

        # (a),(b)... label inside each axis
        ax.set_title(letters[r*n_cols + c], fontsize=16, fontweight="bold", loc="left")

        # Model (solid) + EnKF (dashed): same dataset colour
        ax.plot(T_model, set_model[:, i_global], color=colour, lw=2.6, ls="-")
        ax.plot(T_kf,    set_EnKF[:, i_global],  color=colour, lw=2.6, ls="--")

        # Measurements: dataset marker + same dataset colour
        ax.errorbar(
            T_meas,
            set_meas[:, i_global],
            yerr=set_meas_err[:, i_global] if set_meas_err is not None else None,
            fmt=marker,
            color=colour,
            ecolor="black",
            elinewidth=1.2,
            capsize=2.5,
            markersize=6.0,
            markeredgecolor="black",
            markeredgewidth=0.8,
            alpha=0.95
        )

        # Axis labels (individual, bold, bigger)
        ax.set_xlabel("Time (hours)", fontsize=13, fontweight="bold")
        ax.set_ylabel(axis_name[i_global], fontsize=13, fontweight="bold")

        ax.grid(alpha=0.12)

        # Bigger/bold ticks
        ax.tick_params(axis="both", which="both", labelsize=12, width=1.8, length=4, direction="in")
        for lab in ax.get_xticklabels() + ax.get_yticklabels():
            lab.set_fontweight("bold")

        for spine in ax.spines.values():
            spine.set_linewidth(2.0)

# -----------------------------
# Legend: dataset-per-column (P1 column, P2 column, ...)
# Matplotlib fills column-first, so supply handles grouped by dataset:
# [D1 Model, D1 EnKF, D1 Meas, D2 Model, D2 EnKF, D2 Meas, ...]
# -----------------------------
handles = []
for name in dataset_list:
    c = dataset_colours[name]
    m = dataset_markers[name]
    handles.append(Line2D([0], [0], color=c, lw=2.8, ls="-",  label=f"{name} – Model"))
    handles.append(Line2D([0], [0], color=c, lw=2.8, ls="--", label=f"{name} – EnKF"))
    handles.append(Line2D([0], [0], marker=m, color=c, lw=0,
                          markersize=8, markeredgecolor="black", markeredgewidth=1.0,
                          label=f"{name} – Measurement (validation only)"))

fig.legend(
    handles=handles,
    ncol=len(dataset_list),          # one dataset per column
    loc="lower center",
    bbox_to_anchor=(0.5, -0.2),
    frameon=False,
    fontsize=12.5,
    prop={"weight": "bold"},
    handlelength=2.6,
    handletextpad=0.8,
    columnspacing=2.2,
    labelspacing=0.8
)


save_dir = BASE_RESULTS_DIR / folder_name
save_dir.mkdir(parents=True, exist_ok=True)


fig_name = "all_asn.png"

plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(save_dir / fig_name, dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
import seaborn as sns
from pathlib import Path

sns.set(style="white", context="talk")

# -----------------------------
# What to plot (only Asn)
# -----------------------------
selected_states = {"Asn": 8}
state_key = "Asn"
i_global = selected_states[state_key]

# -----------------------------
# Dataset order (expects 4 -> 2x2)
# -----------------------------
dataset_list = list(DATASETS.keys())  # expects 4 datasets

# -----------------------------
# Colour + marker per dataset
# -----------------------------
default_markers = ["o", "s", "^", "D", "X", "P", "v", "<", ">", "*"]
default_colours = ["tab:orange", "tab:green", "tab:blue", "purple", "tab:red", "tab:brown"]

dataset_colours = {name: default_colours[i % len(default_colours)] for i, name in enumerate(dataset_list)}
dataset_markers = {name: default_markers[i % len(default_markers)] for i, name in enumerate(dataset_list)}

# -----------------------------
# 2x2 Figure layout
# -----------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 9.2), sharex=False)
axes = np.array(axes).reshape(2, 2)

# subplot labels (a), (b), (c), (d)
letters = ["(a)", "(b)", "(c)", "(d)"]

# -----------------------------
# Plotting
# -----------------------------
for idx, name in enumerate(dataset_list[:4]):
    r, c = divmod(idx, 2)
    ax = axes[r, c]

    colour = dataset_colours[name]
    marker = dataset_markers[name]

    set_model = np.asarray(set_model_by_dataset[name], dtype=float)
    set_EnKF  = np.asarray(enkf_results_by_dataset[name], dtype=float)

    data = load_dataset(name)
    set_meas = np.asarray(data["set_meas"], dtype=float)

    set_meas_err = data.get("set_meas_errorbar", data.get("set_meas_error"))
    set_meas_err = None if set_meas_err is None else np.asarray(set_meas_err, dtype=float)

    T_meas = np.asarray(T_meas_by_dataset[name], dtype=float)

    ax.set_title(letters[idx], fontsize=16, fontweight="bold", loc="left")

    # Optional: dataset name in the top-right (comment out if you don't want it)
    ax.text(0.98, 0.98, name, transform=ax.transAxes,
            ha="right", va="top", fontsize=14, fontweight="bold")

    ax.plot(T_model, set_model[:, i_global], color=colour, lw=2.6, ls="-")
    ax.plot(T_kf,    set_EnKF[:,  i_global], color=colour, lw=2.6, ls="--")

    ax.errorbar(
        T_meas,
        set_meas[:, i_global],
        yerr=set_meas_err[:, i_global] if set_meas_err is not None else None,
        fmt=marker,
        color=colour,
        ecolor="black",
        elinewidth=1.2,
        capsize=2.5,
        markersize=6.0,
        markeredgecolor="black",
        markeredgewidth=0.8,
        alpha=0.95
    )

    ax.set_xlabel("Time (hours)", fontsize=13, fontweight="bold")
    ax.set_ylabel(axis_name[i_global], fontsize=13, fontweight="bold")

    ax.grid(alpha=0.12)

    ax.tick_params(axis="both", which="both", labelsize=12, width=1.8, length=4, direction="in")
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight("bold")
    for spine in ax.spines.values():
        spine.set_linewidth(2.0)

# -----------------------------
# Legend: dataset-per-column (P1, P2, P3, P4)
# -----------------------------
handles = []
for name in dataset_list[:4]:
    c = dataset_colours[name]
    m = dataset_markers[name]

    handles.append(Line2D([0], [0], color=c, lw=2.8, ls="-",
                          label=f"{name} – Model"))
    handles.append(Line2D([0], [0], color=c, lw=2.8, ls="--",
                          label=f"{name} – EnKF"))
    handles.append(Line2D([0], [0], marker=m, color=c, lw=0,
                          markersize=8,
                          markeredgecolor="black",
                          markeredgewidth=1.0,
                          label=f"{name} – Validation Measurement"))

fig.legend(
    handles=handles,
    ncol=4,                     # one dataset per column
    loc="lower center",
    bbox_to_anchor=(0.5, -0.14),
    frameon=False,
    fontsize=12.5,
    prop={"weight": "bold"},
    handlelength=2.6,
    handletextpad=0.8,
    columnspacing=2.2,
    labelspacing=0.9
)

# -----------------------------
# Save
# -----------------------------
save_dir = Path(BASE_RESULTS_DIR) / folder_name
save_dir.mkdir(parents=True, exist_ok=True)

fig_name = "all_asn_2x2.png"

# Leave space at bottom for legend
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(save_dir / fig_name, dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

def compute_dimensionless_gramian_multi_dataset_all_states(
    DATASETS,
    state_name,
    T_model,
    dt_model,
    build_schedule,
    volume_results_P,
    state_init_by_dataset,
    model_step,
    measured_state_names,
    epsilon=0.01,
    clip_value=1e-12,
    scale_output=True,
    verbose=False,
    # keep this for backwards compatibility, but it is ignored if include_all_states=True
    exclude_last_n_states=0,
    include_all_states=True,
    # optional: if you also want to allow Asn/Glu to be used as outputs when present in measured_state_names
    exclude_outputs=("Asn", "Glu"),
):
    """
    Dimensionless empirical observability Gramian (diagonal only),
    computed consistently across multiple datasets.

    This version computes the Gramian for all states (including NSDs) when include_all_states=True.

    Returns
    -------
    Wo_by_dataset : dict
        {dataset_name: Wo_diag (np.ndarray of length n_states)}
    measured_indices : list[int]
        Indices of measured states used as outputs
    """

    sns.set(style="white", context="talk")

    # -------------------------
    # State / measurement indices (outputs)
    # -------------------------
    name_to_idx = {n: i for i, n in enumerate(state_name)}
    measured_indices = [
        name_to_idx[n] for n in measured_state_names
        if n in name_to_idx and n not in set(exclude_outputs)
    ]

    if len(measured_indices) == 0:
        raise ValueError("No measured outputs found. Check measured_state_names and exclude_outputs.")

    dataset_list = list(DATASETS.keys())
    n_states = len(state_name)
    dt = float(dt_model)

    # -------------------------
    # Which states to perturb (Gramian states)
    # -------------------------
    if include_all_states:
        perturb_indices = range(n_states)
    else:
        visible_n = n_states - int(exclude_last_n_states)
        perturb_indices = range(visible_n)

    Wo_by_dataset = {}

    for ds_name in tqdm(dataset_list, desc="Dimensionless Gramian (per dataset)", colour="blue"):

        # -------------------------
        # Initial condition
        # -------------------------
        x0_dim = np.array(state_init_by_dataset[ds_name], dtype=float)
        if x0_dim.shape[0] != n_states:
            raise ValueError(f"{ds_name}: initial state length mismatch")

        # -------------------------
        # Dimensionless scaling
        # -------------------------
        state_scale = np.maximum(np.abs(x0_dim), 1.0)
        x0_dimless = x0_dim / state_scale

        output_scale = state_scale[measured_indices]

        # -------------------------
        # Controls & volume
        # -------------------------
        Fin, Fout, Gal_feed, Urd_feed = build_schedule(ds_name)
        V_traj = np.asarray(volume_results_P[ds_name])[1:]

        n_steps = min(len(Fin), len(T_model) - 1)

        def simulate_dimless(x_dimless_init):
            x_dim = x_dimless_init * state_scale
            traj_dim = [x_dim.copy()]

            for k in range(n_steps):
                controls_k = {
                    "Fin": Fin[k],
                    "Fout": Fout[k],
                    "V": V_traj[k],
                    "Gal_feed": Gal_feed[k],
                    "Urd_feed": Urd_feed[k],
                }

                x_dim = model_step(x_dim, T_model[k], controls_k, dt)
                x_dim = np.clip(x_dim, clip_value, None)
                traj_dim.append(x_dim.copy())

            return np.array(traj_dim)

        # -------------------------
        # Gramian diagonal (all states)
        # -------------------------
        Wo_diag = np.zeros(n_states)

        for i in perturb_indices:
            x_plus = x0_dimless.copy()
            x_minus = x0_dimless.copy()

            x_plus[i] += epsilon
            x_minus[i] -= epsilon

            Y_plus = simulate_dimless(x_plus)[:, measured_indices]
            Y_minus = simulate_dimless(x_minus)[:, measured_indices]

            # dimensionless outputs
            Y_plus_dl = Y_plus / output_scale
            Y_minus_dl = Y_minus / output_scale

            delta_y = Y_plus_dl - Y_minus_dl

            if verbose:
                print(f"\n[{ds_name}] Perturb {state_name[i]}")
                print(f"Max |Δy| (dimless):  {np.max(np.abs(delta_y)):.3e}")
                print(f"Mean|Δy| (dimless): {np.mean(np.abs(delta_y)):.3e}")

            gram_raw = np.sum(delta_y**2) * dt
            Wo_diag[i] = gram_raw / (4 * epsilon**2) if scale_output else gram_raw

        Wo_by_dataset[ds_name] = Wo_diag

    return Wo_by_dataset, measured_indices


In [ ]:
# --------------------------------------------------
# User-defined settings
# --------------------------------------------------
epsilon = 0.01      # your model time step
measured_state_names = [
   'Xv', 'mAb', 'Gal', 'Urd', 'Glc', 'Amm', 'Gln', 'Lac'  # adjust to your measured outputs
]

# --------------------------------------------------
# Call Gramian computation
# --------------------------------------------------
Wo_by_dataset, measured_indices = compute_dimensionless_gramian_multi_dataset_all_states(
    DATASETS=DATASETS,
    state_name=state_name,
    T_model=T_model,
    dt_model=dt_model,
    build_schedule=build_schedule,
    volume_results_P=volume_results_P,
    state_init_by_dataset=state_init_by_dataset,
    model_step=model_step,
    measured_state_names=measured_state_names,
    epsilon=epsilon,
    scale_output=True,
    include_all_states=True,
    verbose=False,
)

# --------------------------------------------------
# Example: inspect results for one dataset
# --------------------------------------------------
ds_name = list(Wo_by_dataset.keys())[0]
Wo_diag = Wo_by_dataset[ds_name]

print(f"Dataset: {ds_name}")
for name, val in zip(state_name, Wo_diag):
    print(f"{name:20s}  Wo = {val:.3e}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from pathlib import Path

def plot_gramian_multi_dataset_coloured(
    Wo_by_dataset,
    state_name,
    dataset_list=("P1", "P2", "P3", "P4"),
    dataset_colours=None,
    exclude_states_plot=("Glu",),
    exclude_last_n_states=7,
    asn_name="Asn",
    log_offset=1e-20,
    figsize_per_panel=(4.8, 4.4),
    BASE_RESULTS_DIR=None,
    folder_name=None,
    fig_name="metabolites_Gramian.png",
    dpi=600
):
    """
    Per-dataset empirical observability Gramian plots:
      - One column per dataset
      - Dataset distinguished by colour
      - (a)-(d) above each subplot
      - Asn highlighted (hatched) as unmeasured
      - Legend below

    Plots all states except:
      1) Glu
      2) NSDs (assumed to be the last `exclude_last_n_states`)
    """

    sns.set(style="white", context="talk")

    if dataset_colours is None:
        raise ValueError("Please provide dataset_colours dict.")

    n_states = len(state_name)
    visible_n = n_states - int(exclude_last_n_states)

    exclude_set = set(exclude_states_plot)
    plot_indices = [i for i in range(visible_n) if state_name[i] not in exclude_set]
    plot_labels = [state_name[i] for i in plot_indices]

    # Asn index (if present)
    asn_plot_pos = None
    if asn_name in state_name:
        asn_idx = state_name.index(asn_name)
        if asn_idx in plot_indices:
            asn_plot_pos = plot_indices.index(asn_idx)

    n_cols = len(dataset_list)
    fig, axes = plt.subplots(
        1, n_cols,
        figsize=(figsize_per_panel[0] * n_cols, figsize_per_panel[1]),
        sharey=True
    )
    if n_cols == 1:
        axes = [axes]

    subplot_labels = [f"({chr(ord('a') + i)})" for i in range(n_cols)]

    # ---------------- Plot panels ----------------
    for c, ds_name in enumerate(dataset_list):
        ax = axes[c]
        colour = dataset_colours[ds_name]

        Wo = np.asarray(Wo_by_dataset[ds_name], dtype=float)
        y = Wo[plot_indices]
        y_log = np.log10(y + log_offset)

        x = np.arange(len(plot_indices))
        bars = ax.bar(
            x, y_log,
            color=colour,
            edgecolor="black",
            linewidth=1.2
        )

        if asn_plot_pos is not None:
            bars[asn_plot_pos].set_hatch("//")
            bars[asn_plot_pos].set_linewidth(2.2)

        ax.axhline(0, color="black", linestyle="--", linewidth=1)

        ax.set_title(
            subplot_labels[c],
            loc="left",
            pad=10,
            fontsize=15,
            fontweight="bold"
        )

        ax.set_xticks(x)
        ax.set_xticklabels(plot_labels, rotation=90, fontsize=12, fontweight="bold")

        ax.tick_params(axis="y", labelsize=13)
        for lab in ax.get_yticklabels():
            lab.set_fontweight("bold")

        ax.grid(alpha=0.12)

        for spine in ax.spines.values():
            spine.set_linewidth(2)

        #ax.set_xlabel("State Variable", fontsize=13, fontweight="bold")
        if c == 0:
            ax.set_ylabel("log10(Observability Score)", fontsize=13, fontweight="bold")

    # ---------------- Legend ----------------
    legend_handles = [
        Patch(facecolor=dataset_colours[name], edgecolor="black", label=name)
        for name in dataset_list
    ]

    if asn_plot_pos is not None:
        legend_handles.append(
            Patch(
                facecolor="white",
                edgecolor="black",
                hatch="//",
                linewidth=2.0,
                label="Unmeasured state: Asn"
            )
        )

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=len(legend_handles),
        fontsize=12,
        frameon=False,
        bbox_to_anchor=(0.5, -0.08)
    )

    # ---------------- Save exactly as requested ----------------
    if BASE_RESULTS_DIR is not None and folder_name is not None:
        save_dir = Path(BASE_RESULTS_DIR) / folder_name
        save_dir.mkdir(parents=True, exist_ok=True)

        plt.tight_layout(rect=[0, 0.08, 1, 1])
        plt.savefig(
            save_dir / fig_name,
            dpi=dpi,
            bbox_inches="tight"
        )

    plt.show()
    return fig, axes


In [ ]:
plot_gramian_multi_dataset_coloured(
    Wo_by_dataset=Wo_by_dataset,
    state_name=state_name,
    dataset_list=("P1", "P2", "P3", "P4"),
    dataset_colours=dataset_colours,
    exclude_last_n_states=7,
    BASE_RESULTS_DIR=BASE_RESULTS_DIR,
    folder_name="Gramian",
    fig_name="metabolites_Gramian.png"
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from pathlib import Path

def plot_gramian_multi_dataset_NSDS(
    Wo_by_dataset,
    state_name,
    dataset_list=("P1", "P2", "P3", "P4"),
    dataset_colours=None,
    nsd_last_n_states=7,            # NSDs assumed to be the last block of state_name
    log_offset=1e-20,
    figsize_per_panel=(8, 6),
    BASE_RESULTS_DIR=None,
    folder_name=None,
    fig_name="NSDs_Gramian.png",
    dpi=600
):
    """
    Per-dataset empirical observability Gramian plots for NSDs only:
      - One column per dataset
      - Colour encodes dataset
      - (a)-(d) above each subplot
      - Legend below mapping colour -> dataset
      - Saves using: BASE_RESULTS_DIR / folder_name / fig_name

    Assumes NSDs are the last `nsd_last_n_states` entries in state_name and Wo vectors.
    """

    sns.set(style="white", context="talk")

    if dataset_colours is None:
        raise ValueError("Please provide dataset_colours dict.")

    n_states = len(state_name)
    nsd_last_n_states = int(nsd_last_n_states)
    if nsd_last_n_states <= 0:
        raise ValueError("nsd_last_n_states must be > 0.")
    if nsd_last_n_states > n_states:
        raise ValueError("nsd_last_n_states is larger than the number of states.")

    # NSD indices and labels
    nsd_indices = list(range(n_states - nsd_last_n_states, n_states))
    nsd_labels = [state_name[i] for i in nsd_indices]

    n_cols = len(dataset_list)
    fig, axes = plt.subplots(
        1, n_cols,
        figsize=(figsize_per_panel[0] * n_cols, figsize_per_panel[1]),
        sharey=True
    )
    if n_cols == 1:
        axes = [axes]

    subplot_labels = [f"({chr(ord('a') + i)})" for i in range(n_cols)]

    # ---------------- Plot panels ----------------
    for c, ds_name in enumerate(dataset_list):
        ax = axes[c]

        if ds_name not in Wo_by_dataset:
            raise KeyError(f"{ds_name} not found in Wo_by_dataset.")
        if ds_name not in dataset_colours:
            raise KeyError(f"{ds_name} not found in dataset_colours.")

        colour = dataset_colours[ds_name]
        Wo = np.asarray(Wo_by_dataset[ds_name], dtype=float)

        if Wo.shape[0] != n_states:
            raise ValueError(f"{ds_name}: Wo length ({Wo.shape[0]}) != n_states ({n_states}).")

        y = Wo[nsd_indices]
        y_log = np.log10(y + log_offset)

        x = np.arange(len(nsd_indices))
        ax.bar(
            x, y_log,
            color=colour,
            edgecolor="black",
            linewidth=1.2
        )

        ax.axhline(0, color="black", linestyle="--", linewidth=1)

        ax.set_title(
            subplot_labels[c],
            loc="left",
            pad=10,
            fontsize=15,
            fontweight="bold"
        )

        ax.set_xticks(x)
        ax.set_xticklabels(nsd_labels, rotation=90, fontsize=12, fontweight="bold")

        ax.tick_params(axis="y", labelsize=13)
        for lab in ax.get_yticklabels():
            lab.set_fontweight("bold")

        ax.grid(alpha=0.12)
        for spine in ax.spines.values():
            spine.set_linewidth(2)

       
        if c == 0:
            ax.set_ylabel("log10(Observability Score)", fontsize=13, fontweight="bold")

    # ---------------- Legend (below) ----------------
    legend_handles = [
        Patch(facecolor=dataset_colours[name], edgecolor="black", label=name)
        for name in dataset_list
    ]

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=len(legend_handles),
        fontsize=12,
        frameon=False,
        bbox_to_anchor=(0.5, 0.1)
    )

    # ---------------- Save exactly as requested ----------------
    if BASE_RESULTS_DIR is not None and folder_name is not None:
        save_dir = Path(BASE_RESULTS_DIR) / folder_name
        save_dir.mkdir(parents=True, exist_ok=True)

        plt.tight_layout(rect=[0, 0.08, 1, 1])
        plt.savefig(
            save_dir / fig_name,
            dpi=dpi,
            bbox_inches="tight"
        )

    plt.show()
    return fig, axes


In [ ]:
plot_gramian_multi_dataset_NSDS(
    Wo_by_dataset=Wo_by_dataset,
    state_name=state_name,
    dataset_list=("P1", "P2", "P3", "P4"),
    dataset_colours=dataset_colours,
    nsd_last_n_states=7,
    BASE_RESULTS_DIR=BASE_RESULTS_DIR,
    folder_name= None,
    fig_name="NSDs_Gramian.png"
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from pathlib import Path
from matplotlib.font_manager import FontProperties

def plot_per_dataset_gramian_and_asn_separate_legends(
    Wo_by_dataset,
    state_name,
    axis_name,
    load_dataset,
    set_model_by_dataset,
    enkf_results_by_dataset,
    T_model,
    T_kf,
    T_meas_by_dataset,
    dataset_list=("P1", "P2", "P3", "P4"),
    dataset_colours=None,
    dataset_markers=None,
    # Gramian panel settings
    exclude_last_n_states=7,          # exclude NSDs
    exclude_states_plot=("Glu",),     # exclude Glu
    asn_name="Asn",
    log_offset=1e-20,
    # Saving
    BASE_RESULTS_DIR=None,
    folder_name=None,
    fig_prefix="Gramian_Asn",
    dpi=600,
    figsize=(12.5, 5),
):
    """
    One figure per dataset:
      (a) Metabolite Gramian (log10), with its own legend
      (b) Asn trajectory (Model / EnKF / Measurement), with its own legend
    """

    sns.set(style="white", context="talk")

    if dataset_colours is None or dataset_markers is None:
        raise ValueError("dataset_colours and dataset_markers must be provided.")

    n_states = len(state_name)
    visible_n = n_states - int(exclude_last_n_states)

    # -------- Gramian plot indices (metabolites only) --------
    exclude_set = set(exclude_states_plot)
    gram_indices = [i for i in range(visible_n) if state_name[i] not in exclude_set]
    gram_labels = [state_name[i] for i in gram_indices]

    # Asn index
    if asn_name not in state_name:
        raise ValueError(f"{asn_name} not found in state_name.")
    asn_idx = state_name.index(asn_name)
    asn_gram_pos = gram_indices.index(asn_idx) if asn_idx in gram_indices else None

    # Save directory
    save_dir = None
    if BASE_RESULTS_DIR is not None and folder_name is not None:
        save_dir = Path(BASE_RESULTS_DIR) / folder_name
        save_dir.mkdir(parents=True, exist_ok=True)

    # =========================================================
    # Loop over datasets
    # =========================================================
    for ds_name in dataset_list:

        colour = dataset_colours[ds_name]
        marker = dataset_markers[ds_name]

        fig, (ax_g, ax_a) = plt.subplots(1, 2, figsize=figsize)

        # =====================================================
        # (a) Metabolite Gramian
        # =====================================================
        Wo = np.asarray(Wo_by_dataset[ds_name], dtype=float)
        y = Wo[gram_indices]
        y_log = np.log10(y + log_offset)

        x = np.arange(len(gram_indices))
        bars = ax_g.bar(
            x, y_log,
            color=colour,
            edgecolor="black",
            linewidth=1.2
        )

        # Hatch Asn if present
        if asn_gram_pos is not None:
            bars[asn_gram_pos].set_hatch("//")
            bars[asn_gram_pos].set_linewidth(2.2)

        ax_g.axhline(0, color="black", linestyle="--", linewidth=1)
        ax_g.set_title("(a)", fontsize=16, fontweight="bold", loc="left")
        ax_g.set_xticks(x)
        ax_g.set_xticklabels(gram_labels, rotation=90, fontsize=12, fontweight="bold")
        ax_g.set_xlabel("State Variable", fontsize=13, fontweight="bold", labelpad = 10)
        ax_g.set_ylabel("log10(Observability Score)", fontsize=13, fontweight="bold")
        ax_g.grid(alpha=0.12)

        ax_g.tick_params(axis="y", labelsize=12)
        for lab in ax_g.get_yticklabels():
            lab.set_fontweight("bold")
        for spine in ax_g.spines.values():
            spine.set_linewidth(2)

        # -------- Gramian legend (panel a) --------
        gram_handles = [
            Patch(facecolor=colour, edgecolor="black", label="Measured")
        ]
        if asn_gram_pos is not None:
            gram_handles.append(
                Patch(
                    facecolor= colour,
                    edgecolor="black",
                    hatch="//",
                    linewidth=2.0,
                    label="Unmeasured"
                )
            )

        ax_g.legend(
            handles=gram_handles,
            loc="upper left",
            bbox_to_anchor=(0.12, 1.0),
            frameon=False,
            prop=FontProperties(size=12, weight="bold")
        )

        # =====================================================
        # (b) Asn trajectory
        # =====================================================
        set_model = np.asarray(set_model_by_dataset[ds_name], dtype=float)
        set_EnKF  = np.asarray(enkf_results_by_dataset[ds_name], dtype=float)

        data = load_dataset(ds_name)
        set_meas = np.asarray(data["set_meas"], dtype=float)

        set_meas_err = data.get("set_meas_errorbar", data.get("set_meas_error"))
        set_meas_err = None if set_meas_err is None else np.asarray(set_meas_err, dtype=float)

        T_meas = np.asarray(T_meas_by_dataset[ds_name], dtype=float)

        ax_a.set_title("(b)", fontsize=16, fontweight="bold", loc="left")

        ax_a.plot(T_model, set_model[:, asn_idx], color=colour, lw=2.6, ls="-")
        ax_a.plot(T_kf,    set_EnKF[:,  asn_idx], color=colour, lw=2.6, ls="--")

        ax_a.errorbar(
            T_meas,
            set_meas[:, asn_idx],
            yerr=set_meas_err[:, asn_idx] if set_meas_err is not None else None,
            fmt=marker,
            color=colour,
            ecolor="black",
            elinewidth=1.2,
            capsize=2.5,
            markersize=6.0,
            markeredgecolor="black",
            markeredgewidth=0.8,
            alpha=0.95
        )

        ax_a.set_xlabel("Time (hours)", fontsize=13, fontweight="bold")
        ax_a.set_ylabel(axis_name[asn_idx], fontsize=13, fontweight="bold")
        ax_a.grid(alpha=0.12)

        ax_a.tick_params(axis="both", which="both",
                         labelsize=12, width=1.8, length=4, direction="in")
        for lab in ax_a.get_xticklabels() + ax_a.get_yticklabels():
            lab.set_fontweight("bold")
        for spine in ax_a.spines.values():
            spine.set_linewidth(2)

        # -------- Asn legend (panel b) --------
        asn_handles = [
            Line2D([0], [0], color=colour, lw=2.6, ls="-",  label="Model"),
            Line2D([0], [0], color=colour, lw=2.6, ls="--", label="EnKF"),
            Line2D(
                [0], [0],
                marker=marker,
                color=colour,
                lw=0,
                markersize=7,
                markeredgecolor="black",
                markeredgewidth=1.0,
                label="Measurement"
            )
        ]

        ax_a.legend(
            handles=asn_handles,
            loc="lower left",
            frameon=False,
            prop=FontProperties(size=12, weight="bold")
        )

        # =====================================================
        # Save
        # =====================================================
        plt.tight_layout(rect=[0, 0.08, 1, 1])

        if save_dir is not None:
            fig_name = f"{fig_prefix}_{ds_name}.png"
            plt.savefig(save_dir / fig_name, dpi=dpi, bbox_inches="tight")

        plt.show()
        plt.close(fig)


In [ ]:
plot_per_dataset_gramian_and_asn_separate_legends(
    Wo_by_dataset=Wo_by_dataset,
    state_name=state_name,
    axis_name=axis_name,
    load_dataset=load_dataset,
    set_model_by_dataset=set_model_by_dataset,
    enkf_results_by_dataset=enkf_results_by_dataset,
    T_model=T_model,
    T_kf=T_kf,
    T_meas_by_dataset=T_meas_by_dataset,
    dataset_list=("P1", "P2", "P3", "P4"),
    dataset_colours=dataset_colours,
    dataset_markers=dataset_markers,
    BASE_RESULTS_DIR=BASE_RESULTS_DIR,
    folder_name=folder_name,
    fig_prefix="metabolite_gramian_asn"
)



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from matplotlib.font_manager import FontProperties

def plot_selected_NSDS_per_dataset(
    Wo_by_dataset,
    state_name,
    dataset_list=("P1", "P2", "P3", "P4"),
    dataset_colours=None,
    selected_nsds=("UDPGal", "UDPGlc", "UDPGlcNAc"),
    log_offset=1e-20,
    figsize=(5.6, 4.8),
    BASE_RESULTS_DIR=None,
    folder_name=None,
    fig_prefix="NSDs_Selected_Gramian",
    dpi=600
):
    """
    Creates 4 separate figures (one per dataset) plotting only selected NSDs:
    UDPGal, UDPGlc, UDPGlcNAc (or user-provided selected_nsds).

    Saves each figure to:
      BASE_RESULTS_DIR / folder_name / f"{fig_prefix}_{dataset}.png"
    """

    sns.set(style="white", context="talk")

    if dataset_colours is None:
        raise ValueError("Please provide dataset_colours dict.")

    # Map state names -> indices (global indices in Wo/state vectors)
    name_to_idx = {n: i for i, n in enumerate(state_name)}

    missing = [n for n in selected_nsds if n not in name_to_idx]
    if len(missing) > 0:
        raise ValueError(f"Selected NSDs not found in state_name: {missing}")

    sel_indices = [name_to_idx[n] for n in selected_nsds]
    sel_labels = ["UDP-Gal", "UDP-Glc", "UDP-GlcNAc"]

    # Save directory
    save_dir = None
    if BASE_RESULTS_DIR is not None and folder_name is not None:
        save_dir = Path(BASE_RESULTS_DIR) / folder_name
        save_dir.mkdir(parents=True, exist_ok=True)

    # One figure per dataset
    for ds_name in dataset_list:
        if ds_name not in Wo_by_dataset:
            raise KeyError(f"{ds_name} not found in Wo_by_dataset.")
        if ds_name not in dataset_colours:
            raise KeyError(f"{ds_name} not found in dataset_colours.")

        colour = dataset_colours[ds_name]
        Wo = np.asarray(Wo_by_dataset[ds_name], dtype=float)

        y = Wo[sel_indices]
        y_log = np.log10(y + log_offset)

        fig, ax = plt.subplots(1, 1, figsize=figsize)

        x = np.arange(len(sel_indices))
        ax.bar(
            x, y_log,
            color=colour,
            edgecolor="black",
            linewidth=1.2
        )

        ax.axhline(0, color="black", linestyle="--", linewidth=1)


        ax.set_xticks(x)
        ax.set_xticklabels(sel_labels, rotation=0, fontsize=12, fontweight="bold")
        ax.set_xlabel("State Variable", fontsize=13, fontweight="bold",  labelpad=10)
        
        ax.set_ylabel("log10(Observability Score)", fontsize=13, fontweight="bold")

        ax.grid(alpha=0.12)
        ax.tick_params(axis="y", labelsize=13)
        for lab in ax.get_yticklabels():
            lab.set_fontweight("bold")

        for spine in ax.spines.values():
            spine.set_linewidth(2)

        # dataset legend (single entry)
        ax.legend(
            handles=[plt.Rectangle((0, 0), 1, 1, facecolor=colour, edgecolor="black", label=ds_name)],
            loc="lower left",
            frameon=False,
            prop=FontProperties(size=11, weight="bold")
        )

        # Save
        if save_dir is not None:
            fig_name = f"{fig_prefix}_{ds_name}.png"
            plt.tight_layout(rect=[0, 0.08, 1, 1])
            plt.savefig(save_dir / fig_name, dpi=dpi, bbox_inches="tight")
        else:
            plt.tight_layout(rect=[0, 0.08, 1, 1])

        plt.show()
        plt.close(fig)

    return


In [ ]:
plot_selected_NSDS_per_dataset(
    Wo_by_dataset=Wo_by_dataset,
    state_name=state_name,
    dataset_list=("P1", "P2", "P3", "P4"),
    dataset_colours=dataset_colours,
    BASE_RESULTS_DIR=BASE_RESULTS_DIR,
    folder_name=folder_name,
    fig_prefix="NSDs_UDPGal_UDPGlc_UDPGlcNAc"
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from pathlib import Path
from matplotlib.font_manager import FontProperties

def plot_per_dataset_gramian_and_nsd_shared_legend(
    Wo_by_dataset,
    state_name,
    dataset_list=("P1", "P2", "P3", "P4"),
    dataset_colours=None,
    # Panel (a): extracellular (exclude NSDs)
    exclude_last_n_states=7,
    exclude_states_plot=("Glu",),
    asn_name="Asn",
    log_offset=1e-20,
    # Panel (b): selected NSDs (all unmeasured)
    selected_nsds=("UDPGal", "UDPGlc", "UDPGlcNAc"),
    selected_nsd_labels=("UDP-Gal", "UDP-Glc", "UDP-GlcNAc"),
    nsd_hatch="//",
    # Saving
    BASE_RESULTS_DIR=None,
    folder_name=None,
    fig_prefix="Gramian_with_NSDs",
    dpi=600,
    figsize=(12.5, 5),
):
    """
    One figure per dataset:
      (a) Extracellular metabolite observability Gramian (log10)
      (b) Selected NSD observability Gramian (log10)

    Styling:
    - "Measured" bars: solid fill (no hatch)
    - "Unmeasured" bars: hatched (nsd_hatch, default "//")
    - Asn is hatched in (a) if present
    - ALL NSD bars are hatched in (b) (NSDs are unmeasured)

    Legend:
    - A single shared legend for measured vs unmeasured is placed at the figure level.
    """

    sns.set(style="white", context="talk")

    if dataset_colours is None:
        raise ValueError("dataset_colours must be provided as a dict mapping dataset -> colour.")

    # ---------- indices for panel (a): extracellular states ----------
    n_states = len(state_name)
    visible_n = n_states - int(exclude_last_n_states)

    exclude_set = set(exclude_states_plot)
    gram_indices_a = [i for i in range(visible_n) if state_name[i] not in exclude_set]
    gram_labels_a = [state_name[i] for i in gram_indices_a]

    # Asn hatch position in panel (a)
    asn_gram_pos = None
    if asn_name in state_name:
        asn_idx = state_name.index(asn_name)
        if asn_idx in gram_indices_a:
            asn_gram_pos = gram_indices_a.index(asn_idx)

    # ---------- indices for panel (b): selected NSDs ----------
    name_to_idx = {n: i for i, n in enumerate(state_name)}
    missing = [n for n in selected_nsds if n not in name_to_idx]
    if missing:
        raise ValueError(f"Selected NSDs not found in state_name: {missing}")

    gram_indices_b = [name_to_idx[n] for n in selected_nsds]
    gram_labels_b = list(selected_nsd_labels)

    # ---------- save directory ----------
    save_dir = None
    if BASE_RESULTS_DIR is not None and folder_name is not None:
        save_dir = Path(BASE_RESULTS_DIR) / folder_name
        save_dir.mkdir(parents=True, exist_ok=True)

    # =========================================================
    # Loop over datasets
    # =========================================================
    for ds_name in dataset_list:
        if ds_name not in Wo_by_dataset:
            raise KeyError(f"{ds_name} not found in Wo_by_dataset.")
        if ds_name not in dataset_colours:
            raise KeyError(f"{ds_name} not found in dataset_colours.")

        colour = dataset_colours[ds_name]
        Wo = np.asarray(Wo_by_dataset[ds_name], dtype=float)

        fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=figsize)

        # =====================================================
        # (a) Extracellular metabolite Gramian
        # =====================================================
        y_a = Wo[gram_indices_a]
        y_a_log = np.log10(y_a + log_offset)

        x_a = np.arange(len(gram_indices_a))
        bars_a = ax_a.bar(
            x_a, y_a_log,
            color=colour,
            edgecolor="black",
            linewidth=1.2
        )

        # Hatch Asn only (unmeasured) if present
        if asn_gram_pos is not None:
            bars_a[asn_gram_pos].set_hatch(nsd_hatch)
            bars_a[asn_gram_pos].set_linewidth(2.2)

        ax_a.axhline(0, color="black", linestyle="--", linewidth=1)
        ax_a.set_title("(a)", fontsize=16, fontweight="bold", loc="left")
        ax_a.set_xticks(x_a)
        ax_a.set_xticklabels(gram_labels_a, rotation=90, fontsize=12, fontweight="bold")
        #ax_a.set_xlabel("State Variable", fontsize=13, fontweight="bold", labelpad=10)
        ax_a.set_ylabel("log10(Observability Score)", fontsize=13, fontweight="bold")
        ax_a.grid(alpha=0.12)

        ax_a.tick_params(axis="y", labelsize=12)
        for lab in ax_a.get_yticklabels():
            lab.set_fontweight("bold")
        for spine in ax_a.spines.values():
            spine.set_linewidth(2)

        # =====================================================
        # (b) Selected NSD Gramian (all hatched)
        # =====================================================
        y_b = Wo[gram_indices_b]
        y_b_log = np.log10(y_b + log_offset)

        x_b = np.arange(len(gram_indices_b))
        bars_b = ax_b.bar(
            x_b, y_b_log,
            color=colour,
            edgecolor="black",
            linewidth=1.2
        )

        for bb in bars_b:
            bb.set_hatch(nsd_hatch)
            bb.set_linewidth(2.0)

        ax_b.axhline(0, color="black", linestyle="--", linewidth=1)
        ax_b.set_title("(b)", fontsize=16, fontweight="bold", loc="left")
        ax_b.set_xticks(x_b)
        ax_b.set_xticklabels(gram_labels_b, rotation=0, fontsize=12, fontweight="bold")
        ax_b.set_ylabel("log10(Observability Score)", fontsize=13, fontweight="bold")
        ax_b.grid(alpha=0.12)

        ax_b.tick_params(axis="y", labelsize=12)
        for lab in ax_b.get_yticklabels():
            lab.set_fontweight("bold")
        for spine in ax_b.spines.values():
            spine.set_linewidth(2)

        # =====================================================
        # Shared legend (figure-level): measured vs unmeasured
        # =====================================================
        legend_handles = [
            Patch(facecolor=colour, edgecolor="black", label="Measured"),
            Patch(facecolor=colour, edgecolor="black", hatch=nsd_hatch, linewidth=2.0, label="Unmeasured")
        ]

        fig.legend(
            handles=legend_handles,
            loc="lower center",
            bbox_to_anchor=(0.5, 0.0),
            ncol=2,
            frameon=False,
            prop=FontProperties(size=12, weight="bold")
        )

        # =====================================================
        # Save and show
        # =====================================================
        # Leave extra top space for the shared legend
        plt.tight_layout(rect=[0, 0.06, 1, 0.95])

        if save_dir is not None:
            fig_name = f"{fig_prefix}_{ds_name}.png"
            plt.savefig(save_dir / fig_name, dpi=dpi, bbox_inches="tight")

        plt.show()
        plt.close(fig)


In [ ]:
plot_per_dataset_gramian_and_nsd_shared_legend(
    Wo_by_dataset=Wo_by_dataset,
    state_name=state_name,
    dataset_list=("P1", "P2", "P3", "P4"),
    dataset_colours=dataset_colours,
    exclude_last_n_states=7,                 # NSDs at end of state vector
    exclude_states_plot=("Glu",),             # exclude Glu from extracellular panel
    asn_name="Asn",
    selected_nsds=("UDPGal", "UDPGlc", "UDPGlcNAc"),
    selected_nsd_labels=("UDP-Gal", "UDP-Glc", "UDP-GlcNAc"),
    BASE_RESULTS_DIR=BASE_RESULTS_DIR,
    folder_name="Observability",
    fig_prefix="Observability_Extracellular_vs_NSD",
    dpi=600
)
